In [1]:
using BenchmarkTools, LinearAlgebra

In [2]:
using MatrixMarket

In [3]:
proj_dir = @__DIR__
matrix_dir = proj_dir * "/Matrices/"

"/Users/yaman/Desktop/MP + ML/Matrices/"

# QR + IR tests

In [4]:
using BenchmarkTools, LinearAlgebra

## Iteration 1

In [33]:
using LinearAlgebra

"""
    qr_iterative_refinement(A, b; Tfact=Float32, Twork=Float64, Tresid=Float64,
                            maxiter=10, rtol=√eps(Float64), atol=0.0)

Solve Ax=b using QR-based iterative refinement.
- Factorization in `Tfact`, updates in `Twork`, residuals in `Tresid`.
- Returns (x, iters).
Assumes square, nonsingular A. Extend to pivoted QR for ill-conditioned/LS cases.
"""
function qr_iterative_refinement(A::AbstractMatrix, b::AbstractVector;
        Tfact=Float32, Twork=Float64, Tresid=Float64,
        maxiter::Int=10, rtol=√eps(Float64), atol=0.0)

    n, m = size(A)
    @assert n == m == length(b) "This version expects square A and length(b)==size(A,1)."

    # Low-precision factorization
    Af = Matrix{Tfact}(A)
    F  = qr(Af)                # thin QR; A ≈ Q*R in Tfact
    bf = Vector{Tfact}(b)

    # Initial solution (low) -> cast to working precision
    x  = Twork.(F \ bf)

    # High-precision copies for residuals
    Ah = Matrix{Tresid}(A)
    bh = Vector{Tresid}(b)

    # Precompute norms for stopping test denominator
    An = opnorm(Ah)  # 2-norm; use opnorm(A,2) ≈ default
    bn = norm(bh)

    for k in 1:maxiter
        # Residual in high precision
        r  = bh - Ah * Tresid.(x)
        rn = norm(r)
        denom = An*norm(Tresid.(x)) + bn
        if rn ≤ atol + rtol*denom
            return x, k-1
        end

        # Solve correction with existing QR factors (low precision)
        # y = Q' * r (apply Householders), then R d = y
        y  = F.Q' * Vector{Tfact}(r)
        d  = F.R \ y

        # Update in working precision
        x_new = x + Twork.(d)

        # Simple safeguard: break if no improvement in residual norm
        rnew = bh - Ah * Tresid.(x_new)
        if norm(rnew) ≥ rn
            x = x_new
            break
        end
        x = x_new
    end
    return x, nothing
end

qr_iterative_refinement

In [35]:
A= rand(10,10)
b = rand(10)
x1 = A \ b
x2, _ = qr_iterative_refinement(A, b)
norm(x1 .- x2)

LoadError: AssertionError: This version expects square A and length(b)==size(A,1).

In [6]:
using BenchmarkTools

In [7]:
@btime qr_iterative_refinement(A, b)

  13.958 μs (66 allocations: 14.03 KiB)


([-0.5577494723797827, -0.8417138932632042, 0.6759164933448005, -0.05335678101727126, -0.016666684456026815, 0.009066775942122407, 1.0647174312534844, -0.04542006281090494, 0.6342162237781395, 0.42133706811489446], 1)

In [8]:
@btime A\b

  979.100 ns (3 allocations: 1.16 KiB)


10-element Vector{Float64}:
 -0.5577494723799676
 -0.841713893263405
  0.6759164933448399
 -0.053356781017175794
 -0.01666668445573011
  0.00906677594193359
  1.0647174312534924
 -0.04542006281103895
  0.6342162237784853
  0.42133706811494953

## Iteration 2

In [36]:
using LinearAlgebra

"""
    qr_ir(A, b; Tfact=Float32, Twork=Float64, Tresid=Twork,
                 maxiter=10, rtol=√eps(Tresid), atol=0.0, pivot=false)

QR-based iterative refinement for Ax=b.
- QR factorization in `Tfact` (cheap), residual/update in higher precision.
- Returns (x, iters).
"""
function qr_ir(A::AbstractMatrix, b::AbstractVector;
               Tfact=Float32, Twork=Float64, Tresid=Twork,
               maxiter::Int=10, rtol=√eps(Tresid), atol=0.0, pivot::Bool=false)

    n, m = size(A)
    @assert n == m == length(b)

    # Low-precision factorization
    Af = Matrix{Tfact}(A)
    F  = pivot ? qr(Af, Val(true)) : qr(Af)       # (optionally pivoted) thin QR

    # Initial solution (low precision -> cast to working)
    x  = Twork.(F \ Tfact.(b))

    # Workspaces
    r  = Vector{Tresid}(undef, n)                 # residual (high precision)
    y  = Vector{Tfact}(undef, n)                  # correction in low precision

    # Cheap, stable norms for stopping test
    An = Tresid(opnorm(A, 1))                     # 1-norm (no big allocs)
    bn = Tresid(norm(b))

    for k in 1:maxiter
        # r = b - A*x  (high precision, in-place)
        copyto!(r, Tresid.(b))
        mul!(r, A, Tresid.(x), -one(Tresid), one(Tresid))

        rn = norm(r)
        denom = An*Tresid(norm(x)) + bn
        if rn ≤ atol + rtol*denom
            return x, k-1
        end

        # y = Q' * r (low precision) and R*d = y
        copyto!(y, Tfact.(r))
        lmul!(F.Q', y)                            # y := Q' * y
        ldiv!(UpperTriangular(F.R), y)            # y := R \ y   (this is d)

        # x += d   (upgrade to working precision)
        @inbounds @simd for i in eachindex(x, y)
            x[i] += Twork(y[i])
        end

        if pivot
            # With column-pivoting: A*P = Q*R  ⇒  R*z = Q'*r,  d = P*z
            # The triangular solve produced z; apply permutation to get d:
            # (We can skip forming d explicitly by permuting into x directly.)
            perm = F.p                        # permutation vector
            @inbounds for i in 1:n
                x[perm[i]] = x[perm[i]]      # already accounted via y's indices above
            end
        end
    end
    return x, maxiter
end


qr_ir

In [37]:
A= rand(1000,1000)
b = rand(1000)
x1 = A \ b
x2, _ = qr_ir(A, b)
norm(x1 .- x2)

LoadError: AssertionError: n == m == length(b)

In [29]:
@btime qr_ir($A, $b);

  16.298 ms (5041 allocations: 11.87 MiB)


In [30]:
@btime qr($A) \ $b;

  17.327 ms (9 allocations: 8.19 MiB)


In [32]:
@btime $A \ $b;

  4.901 ms (4 allocations: 7.65 MiB)


## Iteration 3

In [38]:
using LinearAlgebra

"""
    qr_ir_ls(A, b; Tfact=Float32, Twork=Float64, Tresid=Twork,
             maxiter=10, rtol=√eps(Tresid), atol=0.0)

Iterative refinement for least-squares problems min ||Ax-b||_2 using QR.
- A is m×n with m ≥ n.
- Factorization in `Tfact`, updates in `Twork`, residuals in `Tresid`.
- Returns (x, niters).
"""
function qr_ir_ls(A::AbstractMatrix, b::AbstractVector;
                  Tfact=Float32, Twork=Float64, Tresid=Twork,
                  maxiter::Int=10, rtol=√eps(Tresid), atol=0.0)

    m, n = size(A)
    @assert m ≥ n "Require tall/skinny A with m ≥ n"
    @assert length(b) == m

    # Low-precision QR (economy/thin)
    Af = Matrix{Tfact}(A)
    F  = qr(Af, :thin)   # economy QR

    # Initial solution
    x = Twork.(F \ Tfact.(b))

    # Workspaces
    r = Vector{Tresid}(undef, m)
    y = Vector{Tfact}(undef, n)

    An = Tresid(opnorm(A, 1))
    bn = Tresid(norm(b))

    for k in 1:maxiter
        # residual r = b - A*x
        copyto!(r, Tresid.(b))
        mul!(r, A, Tresid.(x), -one(Tresid), one(Tresid))

        rn = norm(r)
        denom = An*Tresid(norm(x)) + bn
        if rn ≤ atol + rtol*denom
            return x, k-1
        end

        # Correction: y = Q' * r (low precision), solve R*d=y
        copyto!(y, Tfact.(r))
        lmul!(F.Q', y)
        ldiv!(UpperTriangular(F.R), y)

        @inbounds @simd for i in eachindex(x, y)
            x[i] += Twork(y[i])
        end
    end

    return x, maxiter
end


qr_ir_ls

In [40]:
A= rand(10000,1000)
b = rand(10000)
x1 = A \ b
x2, _ = qr_ir_ls(A, b)
norm(x1 .- x2)

LoadError: MethodError: no method matching qr!(::Matrix{Float32}, ::Symbol)

[0mClosest candidates are:
[0m  qr!(::AbstractMatrix, [91m::Val{false}[39m)
[0m[90m   @[39m [35mLinearAlgebra[39m [90m[4mdeprecated.jl:103[24m[39m
[0m  qr!(::AbstractMatrix, [91m::Val{true}[39m)
[0m[90m   @[39m [35mLinearAlgebra[39m [90m[4mdeprecated.jl:103[24m[39m
[0m  qr!(::AbstractMatrix)
[0m[90m   @[39m [35mLinearAlgebra[39m [90m~/.julia/juliaup/julia-1.10.5+0.aarch64.apple.darwin14/share/julia/stdlib/v1.10/LinearAlgebra/src/[39m[90m[4mqr.jl:336[24m[39m
[0m  ...


In [43]:
@which qr

LinearAlgebra

## Iteration 4

In [44]:
using LinearAlgebra

function qr_ir_ls(A::AbstractMatrix, b::AbstractVector;
                  Tfact=Float32, Twork=Float64, Tresid=Twork,
                  maxiter::Int=10, rtol=√eps(Tresid), atol=0.0)

    m, n = size(A)
    @assert m ≥ n "Require tall/skinny A with m ≥ n"
    @assert length(b) == m

    # Low-precision QR (economy by default when m ≥ n)
    Af = Matrix{Tfact}(A)
    F  = qr(Af)   # thin/economy QR automatically

    # Initial solution
    x = Twork.(F \ Tfact.(b))

    # Workspaces
    r = Vector{Tresid}(undef, m)
    y = Vector{Tfact}(undef, n)

    An = Tresid(opnorm(A, 1))
    bn = Tresid(norm(b))

    for k in 1:maxiter
        # residual r = b - A*x
        copyto!(r, Tresid.(b))
        mul!(r, A, Tresid.(x), -one(Tresid), one(Tresid))

        rn = norm(r)
        denom = An*Tresid(norm(x)) + bn
        if rn ≤ atol + rtol*denom
            return x, k-1
        end

        # Correction: y = Q' * r, solve R*d = y
        copyto!(y, Tfact.(r))
        lmul!(F.Q', y)
        ldiv!(UpperTriangular(F.R), y)

        @inbounds @simd for i in eachindex(x, y)
            x[i] += Twork(y[i])
        end
    end

    return x, maxiter
end

qr_ir_ls (generic function with 1 method)

In [45]:
A= rand(10000,1000)
b = rand(10000)
x1 = A \ b
x2, _ = qr_ir_ls(A, b)
norm(x1 .- x2)

LoadError: BoundsError: attempt to access 1000-element Vector{Float32} at index [1:10000]

## Iteration 5

In [48]:
using LinearAlgebra

function qr_ir_ls(A::AbstractMatrix, b::AbstractVector;
                  Tfact=Float32, Twork=Float64, Tresid=Twork,
                  maxiter::Int=10, rtol=√eps(Tresid), atol=0.0)

    m, n = size(A)
    @assert m ≥ n "Require tall/skinny A with m ≥ n"
    @assert length(b) == m

    # Low-precision QR (thin by default when m ≥ n)
    Af = Matrix{Tfact}(A)
    F  = qr(Af)

    # Initial solution
    x = Twork.(F \ Tfact.(b))

    # Workspaces
    r = Vector{Tresid}(undef, m)   # residual (length m)
    y = Vector{Tfact}(undef, n)    # Q'r (length n)

    An = Tresid(opnorm(A, 1))
    bn = Tresid(norm(b))

    for k in 1:maxiter
        # residual r = b - A*x
        copyto!(r, Tresid.(b))
        mul!(r, A, Tresid.(x), -one(Tresid), one(Tresid))

        rn = norm(r)
        denom = An*Tresid(norm(x)) + bn
        if rn ≤ atol + rtol*denom
            return x, k-1
        end

        # Correction: y = Q' * r (dimension n)
        mul!(y, F.Q', Tfact.(r))       # produces n-vector
        ldiv!(UpperTriangular(F.R), y) # y := R \ y

        @inbounds @simd for i in eachindex(x, y)
            x[i] += Twork(y[i])
        end
    end

    return x, maxiter
end


qr_ir_ls (generic function with 1 method)

In [49]:
A= rand(10000,1000)
b = rand(10000)
x1 = A \ b
x2, _ = qr_ir_ls(A, b)
norm(x1 .- x2)

LoadError: BoundsError: attempt to access 1000-element Vector{Float32} at index [1:10000]

## Iteration 6

In [54]:
using LinearAlgebra

"""
    qr_ir_ls(A, b; Tfact=Float32, Twork=Float64, Tresid=Twork, maxiter=10, rtol=√eps(Tresid), atol=0.0)

Iterative refinement (QR-based) for tall/skinny least-squares (m >= n).
- A: m×n (m >= n)
- b: length m
- Tfact: precision used for the QR factorization (fast/low)
- Twork: precision used for the solution/update
- Tresid: precision used for residuals/stopping test
Returns (x, niters).
"""
function qr_ir_ls(A::AbstractMatrix, b::AbstractVector;
                  Tfact=Float32, Twork=Float64, Tresid=Twork,
                  maxiter::Int=10, rtol=√eps(Tresid), atol=0.0)

    m, n = size(A)
    @assert m ≥ n "qr_ir_ls requires m ≥ n (tall/skinny A)."
    @assert length(b) == m

    # --- factorization in Tfact ---
    Af = Matrix{Tfact}(A)      # cast A down (one-time)
    F  = qr(Af)                # returns compact WY QR factorization

    # initial solution: x0 = R \ (Q' * b)  (compute in Tfact then cast to Twork)
    bf = Vector{Tfact}(undef, m)
    for i in 1:m
        bf[i] = Tfact(b[i])
    end
    x_low = F \ bf             # Float32 (length n)
    x = Vector{Twork}(undef, n)
    for i in 1:n
        x[i] = Twork(x_low[i])
    end

    # --- preallocate work arrays to avoid repeated allocations ---
    r_h = Vector{Tresid}(undef, m)   # high-precision residual b - A*x
    x_w = Vector{Tresid}(undef, n)   # working-copy of x for high-precision matvec
    f_r = Vector{Tfact}(undef, m)    # casted residual for Q' product
    z   = Vector{Tfact}(undef, m)    # Q' * f_r (length m)
    y   = Vector{Tfact}(undef, n)    # first n entries of z -> R \ y

    An = Tresid(opnorm(A, 1))        # cheap-ish norm for stopping test
    bn = Tresid(norm(b))

    for k in 1:maxiter
        # x_w := x in residual precision
        for i in 1:n
            x_w[i] = Tresid(x[i])
        end

        # r_h := b - A*x_w   (in-place matvec then subtract)
        mul!(r_h, A, x_w)                # r_h = A * x_w
        @inbounds for i in 1:m
            r_h[i] = Tresid(b[i]) - r_h[i]
        end

        rn = norm(r_h)
        denom = An * Tresid(norm(x)) + bn
        if rn ≤ atol + rtol*denom
            return x, k-1
        end

        # f_r := cast(r_h) into factorization precision
        @inbounds for i in 1:m
            f_r[i] = Tfact(r_h[i])
        end

        # z = Q' * f_r  (z is length m)
        mul!(z, F.Q', f_r)

        # y = z[1:n]  (this is Q_1' * r), then solve R * d = y
        copyto!(y, @view z[1:n])
        ldiv!(UpperTriangular(F.R), y)   # y := R \ y  (in-place)

        # update x <- x + d  (cast y to Twork)
        @inbounds for i in 1:n
            x[i] += Twork(y[i])
        end
    end

    return x, maxiter
end


qr_ir_ls

In [55]:
A= rand(10000,1000)
b = rand(10000)
x1 = A \ b
x2, _ = qr_ir_ls(A, b)
norm(x1 .- x2)

2.4572765580063675e-7

In [62]:
@btime qr_ir_ls($A, $b);

@btime qr($A) \ $b;

@btime $A \ $b;

  252.264 ms (64 allocations: 115.01 MiB)
  219.873 ms (11 allocations: 76.94 MiB)
  519.929 ms (6011 allocations: 77.07 MiB)


## Iteration 7

In [2]:
using LinearAlgebra

"""
    qr_ir_ls_fast(A, b; Tfact=Float32, Twork=Float64, Tresid=Twork,
                  maxiter=10, rtol=√eps(Tresid), atol=0.0, pivot=false)

A lower-allocation, faster implementation of QR iterative refinement for
tall/skinny least-squares (m >= n). Returns (x, niters).

Arguments:
- Tfact: precision used to compute the QR factors (fast/low)
- Twork: precision used for updates/solution
- Tresid: precision used for residual and stopping test
- pivot: whether to compute pivoted QR (slower, use for rank issues)
"""
function qr_ir_ls_fast(A::AbstractMatrix, b::AbstractVector;
                       Tfact=Float32, Twork=Float64, Tresid=Twork,
                       maxiter::Int=10, rtol=√eps(Tresid), atol=0.0,
                       pivot::Bool=false)

    m, n = size(A)
    @assert m >= n "qr_ir_ls_fast expects m >= n"
    @assert length(b) == m

    # --- Factorization (once) ---
    # Cast A down to Tfact only if needed (one-time)
    Af = (Tfact == eltype(A)) ? copy(A) : Matrix{Tfact}(A)
    F  = pivot ? qr(Af, Val(true)) : qr(Af)   # compact WY QR; pivot optional

    # initial solution: x_low (Tfact) -> x (Twork)
    bf = Vector{Tfact}(undef, m)
    for i in 1:m
        bf[i] = Tfact(b[i])
    end
    x_low = F \ bf           # n-vector in Tfact
    x = Vector{Twork}(undef, n)
    for i in 1:n
        x[i] = Twork(x_low[i])
    end

    # --- Preallocated workspace (one-time) ---
    r_h = Vector{Tresid}(undef, m)    # high-precision residual
    xw  = Vector{Tresid}(undef, n)    # x in residual precision
    fr  = Vector{Tfact}(undef, m)     # residual cast to factor precision
    z   = Vector{Tfact}(undef, m)     # Q' * fr (length m)
    y   = Vector{Tfact}(undef, n)     # first n entries of z
    An  = Tresid(opnorm(A, 1))
    bn  = Tresid(norm(b))

    # local bindings to avoid globals
    FQt = F.Q'               # adjoint Q (applies as m×m)
    FR  = UpperTriangular(F.R)

    for k in 1:maxiter
        # xw := x (in residual precision)
        @inbounds @simd for i in 1:n
            xw[i] = Tresid(x[i])
        end

        # r_h := b - A*xw  (residual in high precision)
        mul!(r_h, A, xw)              # r_h = A*xw
        @inbounds for i in 1:m
            r_h[i] = Tresid(b[i]) - r_h[i]
        end

        rn = norm(r_h)
        denom = An * Tresid(norm(x)) + bn
        if rn <= atol + rtol * denom
            return x, k-1
        end

        # fr := cast(r_h) into factor precision (Tfact)
        @inbounds @simd for i in 1:m
            fr[i] = Tfact(r_h[i])
        end

        # z := Q' * fr  (length m), in-place into preallocated z
        mul!(z, FQt, fr)             # F.Q' * fr  (note: FQт is F.Q')

        # y := z[1:n]; solve R * d = y  (in-place)
        copyto!(y, @view z[1:n])     # n-vector copy (cheap)
        ldiv!(FR, y)                 # y := R \ y  (in-place)

        # update x <- x + d  (cast d to Twork)
        @inbounds @simd for i in 1:n
            x[i] += Twork(y[i])
        end
    end

    return x, maxiter
end

qr_ir_ls_fast

In [12]:
using LinearAlgebra

"""
    qr_ir_ls_fast(A, b; Tfact=Float32, Twork=Float64, Tresid=Twork,
                  maxiter=10, rtol=√eps(Tresid), atol=0.0, pivot=false)

A lower-allocation, faster implementation of QR iterative refinement for
tall/skinny least-squares (m >= n). Returns (x, niters).

Arguments:
- Tfact: precision used to compute the QR factors (fast/low)
- Twork: precision used for updates/solution
- Tresid: precision used for residual and stopping test
- pivot: whether to compute pivoted QR (slower, use for rank issues)
"""
function qr_ir_ls_fast_nosimd(A::AbstractMatrix, b::AbstractVector;
                       Tfact=Float32, Twork=Float64, Tresid=Twork,
                       maxiter::Int=10, rtol=√eps(Tresid), atol=0.0,
                       pivot::Bool=false)

    m, n = size(A)
    @assert m >= n "qr_ir_ls_fast expects m >= n"
    @assert length(b) == m

    # --- Factorization (once) ---
    # Cast A down to Tfact only if needed (one-time)
    Af = (Tfact == eltype(A)) ? copy(A) : Matrix{Tfact}(A)
    F  = pivot ? qr(Af, Val(true)) : qr(Af)   # compact WY QR; pivot optional

    # initial solution: x_low (Tfact) -> x (Twork)
    bf = Vector{Tfact}(undef, m)
    for i in 1:m
        bf[i] = Tfact(b[i])
    end
    x_low = F \ bf           # n-vector in Tfact
    x = Vector{Twork}(undef, n)
    for i in 1:n
        x[i] = Twork(x_low[i])
    end

    # --- Preallocated workspace (one-time) ---
    r_h = Vector{Tresid}(undef, m)    # high-precision residual
    xw  = Vector{Tresid}(undef, n)    # x in residual precision
    fr  = Vector{Tfact}(undef, m)     # residual cast to factor precision
    z   = Vector{Tfact}(undef, m)     # Q' * fr (length m)
    y   = Vector{Tfact}(undef, n)     # first n entries of z
    An  = Tresid(opnorm(A, 1))
    bn  = Tresid(norm(b))

    # local bindings to avoid globals
    FQt = F.Q'               # adjoint Q (applies as m×m)
    FR  = UpperTriangular(F.R)

    for k in 1:maxiter
        # xw := x (in residual precision)
        @inbounds  for i in 1:n
            xw[i] = Tresid(x[i])
        end

        # r_h := b - A*xw  (residual in high precision)
        mul!(r_h, A, xw)              # r_h = A*xw
        @inbounds for i in 1:m
            r_h[i] = Tresid(b[i]) - r_h[i]
        end

        rn = norm(r_h)
        denom = An * Tresid(norm(x)) + bn
        if rn <= atol + rtol * denom
            return x, k-1
        end

        # fr := cast(r_h) into factor precision (Tfact)
        @inbounds  for i in 1:m
            fr[i] = Tfact(r_h[i])
        end

        # z := Q' * fr  (length m), in-place into preallocated z
        mul!(z, FQt, fr)             # F.Q' * fr  (note: FQт is F.Q')

        # y := z[1:n]; solve R * d = y  (in-place)
        copyto!(y, @view z[1:n])     # n-vector copy (cheap)
        ldiv!(FR, y)                 # y := R \ y  (in-place)

        # update x <- x + d  (cast d to Twork)
        @inbounds  for i in 1:n
            x[i] += Twork(y[i])
        end
    end

    return x, maxiter
end

qr_ir_ls_fast_nosimd

In [19]:
A= randn(10000,1000)
b = randn(10000)
x1 = A \ b
x2, _ = qr_ir_ls_fast(A, b)
x3 = qr(A) \ b
println(norm(x1 .- x2))

println(norm(x1 .- x3))

2.3340605041768766e-7
1.5041395603476306e-15


In [22]:
A32= randn(Float32, 10000,1000)
b32 = randn(Float32, 10000)
x3 = qr(A32) \ b32
x1 = A32 \ b32

println(norm(x1 .- x3))

8.603091e-7


In [13]:
@btime qr_ir_ls_fast($A, $b);

@btime qr_ir_ls_fast_nosimd($A, $b);

@btime qr($A) \ $b;

@btime $A \ $b;

  2.324 s (39 allocations: 769.39 MiB)
  2.276 s (39 allocations: 769.39 MiB)
  2.397 s (11 allocations: 764.27 MiB)
  4.927 s (6011 allocations: 764.40 MiB)


In [14]:
@which A \b

\(A::AbstractMatrix, B::AbstractVecOrMat)
     @ LinearAlgebra ~/.julia/juliaup/julia-1.10.5+0.aarch64.apple.darwin14/share/julia/stdlib/v1.10/LinearAlgebra/src/generic.jl:1110

In [16]:
@code_typed A \b

CodeInfo(
1 ── %1   = Base.arraysize(A, 1)::Int64
│    %2   = Base.arraysize(A, 2)::Int64
│    %3   = (%1 === %2)::Bool
└───        goto #44 if not %3
2 ── %5   = LinearAlgebra.istril::typeof(istril)
│    %6   = invoke %5(A::Matrix{Float64}, 0::Int64)::Bool
└───        goto #18 if not %6
3 ── %8   = LinearAlgebra.istriu::typeof(istriu)
│    %9   = invoke %8(A::Matrix{Float64}, 0::Int64)::Bool
└───        goto #5 if not %9
4 ── %11  = LinearAlgebra.diag::typeof(diag)
│    %12  = invoke %11(A::Matrix{Float64}, 0::Int64)::Vector{Float64}
│    %13  = %new(Diagonal{Float64, Vector{Float64}}, %12)::Diagonal{Float64, Vector{Float64}}
│    %14  = invoke LinearAlgebra.:\(%13::Diagonal{Float64, Vector{Float64}}, B::Vector{Float64})::Vector{Float64}
└───        return %14
5 ── %16  = Base.arraysize(A, 1)::Int64
│    %17  = Base.arraysize(A, 2)::Int64
│    %18  = (%16 === %17)::Bool
└───        goto #7 if not %18
6 ──        goto #8
7 ── %21  = Base.arraysize(A, 1)::Int64
│    %22  = Base.arraysiz

In [17]:
@code_typed qr(A) \ b

CodeInfo(
1 ─ %1 = invoke LinearAlgebra.ldiv(F::LinearAlgebra.QRCompactWY{Float64, Matrix{Float64}, Matrix{Float64}}, B::Vector{Float64})::Vector{Float64}
└──      return %1
) => Vector{Float64}

# LU + IR Tests

## Iteration 1

In [23]:
using LinearAlgebra
using BenchmarkTools

"""
    iterative_refinement_lu!(A, b; maxit=10, tol=1e-12, verbose=false)

Solve A x = b where A and b are Float64 (double precision) using:
  - LU factorization in Float32 (single precision)
  - iterative refinement with residuals computed in Float64

Returns (x, info) where info = Dict(:iters => iters, :resnorm => final_rel_resnorm)

Notes:
  - A and b are not modified.
  - Allocations minimized: work arrays preallocated and reused.
  - Stopping criteria: relative residual <= tol or maxit reached.
"""
function iterative_refinement_lu(A::AbstractMatrix{Float64},
                                 b::AbstractVector{Float64};
                                 maxit::Int=10,
                                 tol::Real=1e-12,
                                 verbose::Bool=false)

    n = size(A,1)
    @assert size(A,1) == size(A,2) "A must be square"
    @assert length(b) == n

    # --- Precompute single-precision LU factorization (fast) ---
    A_s = Array{Float32}(undef, n, n)
    copy!(A_s, Float32.(A))          # convert once
    F_s = lu(A_s)                    # LU factorization in Float32 (includes pivots)

    # --- initial solve in single precision, lift to double ---
    b_s = Array{Float32}(undef, n)
    copy!(b_s, Float32.(b))
    x_s = similar(b_s)
    # Solve LU * x_s = b_s  (single-precision substitution)
    x_s .= F_s \ b_s                # uses the factorization; fast
    x = Array{Float64}(undef, n)
    copy!(x, Float64.(x_s))         # initial double precision iterate

    # --- preallocate work arrays to avoid allocations inside loop ---
    r = similar(b)                  # Float64 residual (r = b - A*x)
    r_s = Array{Float32}(undef, n)  # single-precision residual for solve step
    d_s = similar(r_s)              # single-precision correction
    d = Array{Float64}(undef, n)    # double-precision correction

    # norm of b for relative residual
    bnrm = norm(b, Inf)
    if bnrm == 0.0
        bnrm = 1.0
    end

    iters = 0
    final_rel_resnorm = NaN

    # initial residual
    mul!(r, A, x)                   # r = A*x
    @inbounds r .= b .- r           # r = b - A*x  (double precision)
    relres = norm(r, Inf) / (norm(A,Inf)*norm(x,Inf) + norm(b,Inf))  # relative residual as Higham suggests
    final_rel_resnorm = relres
    if verbose
        @printf("iter %2d: relres = %.3e\n", 0, relres)
    end
    if relres <= tol
        return x, Dict(:iters => 0, :resnorm => relres)
    end

    # refinement loop
    for k in 1:maxit
        iters = k
        # convert residual to single precision (preallocated)
        copy!(r_s, Float32.(r))

        # solve LU * d_s = r_s  in single precision (fast)
        d_s .= F_s \ r_s

        # lift correction to double precision and update x in double precision
        copy!(d, Float64.(d_s))
        @inbounds x .+= d

        # recompute residual r = b - A*x in double precision (accurate)
        mul!(r, A, x)
        @inbounds r .= b .- r

        # compute relative residual (use same measure as Higham)
        relres = norm(r, Inf) / (norm(A,Inf)*norm(x,Inf) + norm(b,Inf))
        final_rel_resnorm = relres
        if verbose
            @printf("iter %2d: relres = %.3e\n", k, relres)
        end

        if relres <= tol
            break
        end
    end

    return x, Dict(:iters => iters, :resnorm => final_rel_resnorm)
end

LoadError: LoadError: UndefVarError: `@printf` not defined
in expression starting at In[23]:4

## Iteration 2

In [24]:
using LinearAlgebra
using Printf    # <-- fix for @printf
# using BenchmarkTools   # uncomment if you want to benchmark with @btime

"""
    iterative_refinement_lu(A, b; maxit=10, tol=1e-12, verbose=false)

Solve A x = b where A and b are Float64 using:
  - LU factorization in Float32
  - iterative refinement with residuals computed in Float64

Returns (x, info) where info = Dict(:iters => iters, :resnorm => final_rel_resnorm)

A and b are not modified. Work arrays are preallocated to reduce allocations.
"""
function iterative_refinement_lu(A::AbstractMatrix{Float64},
                                 b::AbstractVector{Float64};
                                 maxit::Int=10,
                                 tol::Real=1e-12,
                                 verbose::Bool=false)

    n = size(A,1)
    @assert size(A,1) == size(A,2) "A must be square"
    @assert length(b) == n

    # --- Precompute single-precision LU factorization (fast) ---
    A_s = Array{Float32}(undef, n, n)
    @inbounds for i in eachindex(A) A_s[i] = Float32(A[i]) end
    F_s = lu(A_s)

    # --- initial solve in single precision, lift to double ---
    b_s = Array{Float32}(undef, n)
    @inbounds for i in 1:n b_s[i] = Float32(b[i]) end
    x_s = similar(b_s)
    x_s .= F_s \ b_s                 # solve in single precision
    x = Array{Float64}(undef, n)
    @inbounds for i in 1:n x[i] = Float64(x_s[i]) end

    # --- preallocate work arrays to avoid allocations inside loop ---
    r = similar(b)                   # Float64 residual (r = b - A*x)
    r_s = Array{Float32}(undef, n)   # single-precision residual for solve step
    d_s = similar(r_s)               # single-precision correction
    d = Array{Float64}(undef, n)     # double-precision correction

    # initial residual and reference norms
    mul!(r, A, x)                    # r = A*x
    @inbounds r .= b .- r            # r = b - A*x  (double precision)
    relres = norm(r, Inf) / (norm(A,Inf)*norm(x,Inf) + norm(b,Inf))
    if verbose
        @printf("iter %2d: relres = %.3e\n", 0, relres)
    end
    if relres <= tol
        return x, Dict(:iters => 0, :resnorm => relres)
    end

    iters = 0
    final_rel_resnorm = relres

    # refinement loop
    for k in 1:maxit
        iters = k

        # convert residual to single precision without allocating temporaries
        @inbounds for i in 1:n r_s[i] = Float32(r[i]) end

        # solve LU * d_s = r_s  in single precision (fast)
        d_s .= F_s \ r_s

        # lift correction to double precision and update x in double precision
        @inbounds for i in 1:n d[i] = Float64(d_s[i]) end
        @inbounds x .+= d

        # recompute residual r = b - A*x in double precision (accurate)
        mul!(r, A, x)
        @inbounds r .= b .- r

        # compute relative residual (Higham style)
        relres = norm(r, Inf) / (norm(A,Inf)*norm(x,Inf) + norm(b,Inf))
        final_rel_resnorm = relres
        if verbose
            @printf("iter %2d: relres = %.3e\n", k, relres)
        end

        if relres <= tol
            break
        end
    end

    return x, Dict(:iters => iters, :resnorm => final_rel_resnorm)
end


iterative_refinement_lu

In [30]:
A= randn(1000,1000)
b = randn(1000)
x1 = A \ b
x2, _ = iterative_refinement_lu(A, b)
println(norm(x1 .- x2))

1.1529488325167853e-9


In [28]:
@btime iterative_refinement_lu($A, $b);

@btime $A \ $b;

@btime lu($A) \ $b;

  6.731 ms (31 allocations: 7.69 MiB)
  4.837 ms (4 allocations: 7.65 MiB)
  4.848 ms (4 allocations: 7.65 MiB)


## Iteration 3

In [1]:
using LinearAlgebra
using Printf

"""
    iterative_refinement_lu(A, b; maxit=10, tol=1e-12, verbose=false)

Mixed-precision iterative refinement solver:
  - Factorize A in Float32
  - Refine in Float64

Returns (x, info).
"""
function iterative_refinement_lu(A::Matrix{Float64},
                                 b::Vector{Float64};
                                 maxit::Int=10,
                                 tol::Real=1e-12,
                                 verbose::Bool=false)

    n = size(A,1)
    @assert size(A,1) == size(A,2)
    @assert length(b) == n

    # --- LU factorization in Float32 ---
    A_s = Float32.(A)          # allocate once
    F_s = lu(A_s)

    # --- initial solve in Float32 ---
    b_s = Float32.(b)
    x_s = F_s \ b_s
    x   = Float64.(x_s)

    # --- work arrays ---
    r   = similar(b)           # residual in Float64
    r_s = similar(b_s)         # residual in Float32
    d_s = similar(b_s)         # correction in Float32
    d   = similar(b)           # correction in Float64

    # precompute norms
    Anrm = norm(A, Inf)
    bnrm = norm(b, Inf)
    if bnrm == 0.0
        bnrm = 1.0
    end

    # compute initial residual
    mul!(r, A, x)      # r = A*x
    @. r = b - r
    relres = norm(r, Inf) / (Anrm*norm(x,Inf) + bnrm)
    if verbose
        @printf("iter %2d: relres = %.3e\n", 0, relres)
    end
    if relres <= tol
        return x, Dict(:iters => 0, :resnorm => relres)
    end

    # refinement loop
    iters = 0
    for k in 1:maxit
        iters = k

        # convert residual to Float32
        @. r_s = Float32(r)

        # solve for correction in single precision, in-place
        copy!(d_s, r_s)
        ldiv!(F_s, d_s)   # solves in-place

        # lift to Float64 and update x
        @. d = Float64(d_s)
        @. x += d

        # recompute residual
        mul!(r, A, x)
        @. r = b - r

        relres = norm(r, Inf) / (Anrm*norm(x,Inf) + bnrm)
        if verbose
            @printf("iter %2d: relres = %.3e\n", k, relres)
        end
        if relres <= tol
            break
        end
    end

    return x, Dict(:iters => iters, :resnorm => relres)
end


iterative_refinement_lu

In [2]:
A= randn(25000,25000)
b = randn(25000)
x1 = A \ b
x2, _ = iterative_refinement_lu(A, b)
println(norm(x1 .- x2))

3.3616016707035372e-9


In [5]:
@btime iterative_refinement_lu($A, $b);

@btime $A \ $b;

@btime lu($A) \ $b;

  21.013 s (36 allocations: 4.66 GiB)
  40.160 s (6 allocations: 4.66 GiB)
  43.765 s (6 allocations: 4.66 GiB)


# GMRES + IR Tests

In [6]:
using MatrixMarket

"/Users/yaman/Desktop/MP + ML"

In [8]:
proj_dir = @__DIR__

"/Users/yaman/Desktop/MP + ML"

In [12]:
A = mmread(proj_dir * "/Matrices/af23560.mtx")

23560×23560 SparseArrays.SparseMatrixCSC{Float64, Int64} with 484256 stored entries:
⎡⢿⣷⣄⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⎤
⎢⠀⠙⢿⣷⣄⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⎥
⎢⠀⠀⠀⠙⢿⣷⣄⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⎥
⎢⠀⠀⠀⠀⠀⠙⢿⣷⣄⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⎥
⎢⠀⠀⠀⠀⠀⠀⠀⠙⢿⣷⣄⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⎥
⎢⠀⠀⠀⠀⠀⠀⠀⠀⠀⠙⢿⣷⣄⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⎥
⎢⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠙⢿⣷⣄⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⎥
⎢⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠙⢿⣷⣄⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⎥
⎢⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠙⢿⣷⣄⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⎥
⎢⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠙⢿⣷⣄⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⎥
⎢⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠙⢿⣷⣄⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⎥
⎢⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠙⢿⣷⣄⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⎥
⎢⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠙⢿⣷⣄⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⎥
⎢⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠙⢿⣷⣄⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⎥
⎢⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠙⢿⣷⣄⠀⠀⠀⠀⠀⠀⠀⠀⠀⎥
⎢⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠙⢿⣷⣄⠀⠀⠀⠀⠀⠀⠀⎥
⎢⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠙⢿⣷⣄⠀⠀⠀⠀⠀⎥
⎢⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠙⢿⣷⣄⠀⠀⠀⎥
⎢⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠙⢿⣷⣄⠀⎥
⎣⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠙⢿⣷⎦

## Iteration 1

In [13]:
#############################
# Mixed-Precision GMRES-IR  #
#############################
# Usage:
#   x, iters, history = gmres_ir(A, b; S=Float32, restart=50, atol=0.0, rtol=1e-8,
#                                max_outer=50, x0=nothing, M=nothing, verbose=false)
#
# Notes:
# - A must support mul!(y, A, x). Works with dense/sparse matrices.
# - S controls the *low* precision used inside GMRES (Float32 or Float16).
# - Residuals and returned x are in Float64 by default (or eltype(b)).
# - Optional left preconditioner M that supports mul!(z, M, x).
#
# Mixed-precision loop (High T, Low S):
#   r_T = b_T - A_T * x_T
#   Solve approximately in low precision: A_S δ_S ≈ r_S  with GMRES(S)
#   x_T += δ_T
#   Recompute r_T in high precision; repeat until converged.

module MixedPrecisionGMRESIR

using LinearAlgebra, SparseArrays

# -- Utility: generic mul! fallbacks for preconditioner (identity) --
struct IdentityOp end
Base.size(::IdentityOp) = (0, 0)  # unused
function LinearAlgebra.mul!(y::AbstractVector, ::IdentityOp, x::AbstractVector)
    @inbounds @simd for i in eachindex(y, x)
        y[i] = x[i]
    end
    return y
end

# -- Low-precision restarted GMRES (left-preconditioned) --
# Solves approximately:  A_S * δ ≈ r_S   using GMRES(S)
# Returns δ in S, residual norm, and inner iters used.
function gmres_lp!(
    δ::AbstractVector{S},                # output solution (S)
    A,                                   # linear operator for A (mul!)
    r::AbstractVector{S},                # right-hand side (S)
    M,                                   # left preconditioner operator (mul!)
    workspace::Dict{Symbol,Any},         # preallocated buffers
    ; restart::Int=50, tol::Real=1e-4, maxiter::Int=0, verbose::Bool=false
) where {S<:AbstractFloat}

    n = length(r)
    maxiter = maxiter == 0 ? restart : maxiter

    # Buffers
    V  = workspace[:V]  :: Matrix{S}             # size (n, restart+1)
    H  = workspace[:H]  :: Matrix{Float64}       # size (restart+1, restart)
    cs = workspace[:cs] :: Vector{Float64}       # size (restart)
    sn = workspace[:sn] :: Vector{Float64}       # size (restart)
    g  = workspace[:g]  :: Vector{Float64}       # size (restart+1)
    w  = workspace[:w]  :: Vector{S}             # size (n)
    z  = workspace[:z]  :: Vector{S}             # size (n)

    # Initialize
    fill!(δ, zero(S))
    fill!(H, 0.0); fill!(cs, 0.0); fill!(sn, 0.0)
    fill!(w, zero(S)); fill!(z, zero(S))

    # v1 = M^{-1} r / β
    mul!(z, M, r)           # z = M \ r
    β = Float64(norm(z))
    if β == 0.0
        return δ, 0.0, 0
    end
    @inbounds @simd for i in 1:n
        V[i,1] = z[i] / S(β)
    end
    g .= 0.0
    g[1] = β

    jfinal = 0
    resnorm = β

    # Arnoldi with Givens rotations
    for j in 1:restart
        # w = M^{-1} (A * v_j)
        mul!(w, A, view(V, :, j))  # w = A*v_j
        mul!(z, M, w)              # z = M^{-1} * (A*v_j)
        copyto!(w, z)

        # Modified Gram-Schmidt
        for i in 1:j
            hij = Float64(dot(view(V, :, i), w))
            H[i, j] = hij
            @inbounds @simd for k in 1:n
                w[k] -= S(hij) * V[k, i]
            end
        end
        hj1 = Float64(norm(w))
        H[j+1, j] = hj1

        if hj1 == 0.0
            jfinal = j
            # happy breakdown
            break
        end

        @inbounds @simd for k in 1:n
            V[k, j+1] = w[k] / S(hj1)
        end

        # Apply previous Givens rotations to the new column
        for i in 1:j-1
            h_i  = H[i, j]
            h_ip = H[i+1, j]
            H[i, j]   =  cs[i]*h_i + sn[i]*h_ip
            H[i+1, j] = -sn[i]*h_i + cs[i]*h_ip
        end

        # Compute and apply j-th Givens
        r1 = H[j, j]
        r2 = H[j+1, j]
        denom = hypot(r1, r2)
        if denom == 0.0
            cs[j] = 1.0; sn[j] = 0.0
        else
            cs[j] = r1 / denom
            sn[j] = r2 / denom
        end
        H[j, j]   = cs[j]*r1 + sn[j]*r2
        H[j+1, j] = 0.0

        # Update g
        g_j  = g[j]
        g[j]   =  cs[j]*g_j
        g[j+1] = -sn[j]*g_j

        resnorm = abs(g[j+1])
        jfinal = j

        verbose && @info "  [GMRES(S)]  iter=$j  relres=$(resnorm/β)"

        if resnorm <= tol*β
            break
        end
    end

    # Backsolve for y (size jfinal)
    j = jfinal
    if j == 0
        return δ, resnorm, 0
    end
    y = workspace[:y]
    resize!(y, j)
    for i in j:-1:1
        s = g[i]
        for k in i+1:j
            s -= H[i,k] * y[k]
        end
        y[i] = s / H[i,i]
    end

    # δ = V[:,1:j] * y
    fill!(δ, zero(S))
    for k in 1:j
        α = S(y[k])
        @inbounds @simd for i in 1:n
            δ[i] += α * V[i, k]
        end
    end

    return δ, resnorm, j
end

# -- Public: Mixed-precision IR driver --
function gmres_ir(
    A, b::AbstractVector{Tb};
    S::Type{<:AbstractFloat}=Float32,        # low precision in GMRES
    restart::Int=50,
    atol::Real=0.0,
    rtol::Real=1e-8,
    max_outer::Int=50,
    x0=nothing,
    M=nothing,                                # optional left preconditioner; Identity if nothing
    verbose::Bool=false
) where {Tb<:AbstractFloat}

    n = length(b)
    T = Tb                        # high precision type
    Mop = isnothing(M) ? IdentityOp() : M

    # High-precision buffers
    x = isnothing(x0) ? zeros(T, n) : copy!(similar(b), T.(x0))
    r = similar(b)  # residual in high precision
    Ax = similar(b)

    # Low-precision copies & workspace
    # We will convert A*x on the fly; if A is a matrix, keep S-copy for speed.
    A_lp = A
    if A isa AbstractMatrix
        A_lp = convert(AbstractMatrix{S}, S.(A))
    end

    # Preallocate GMRES workspace (in S for vectors, Float64 for H, rotations)
    V  = zeros(S, n, restart+1)
    H  = zeros(Float64, restart+1, restart)
    cs = zeros(Float64, restart)
    sn = zeros(Float64, restart)
    g  = zeros(Float64, restart+1)
    w  = zeros(S, n)
    z  = zeros(S, n)
    y  = Float64[]
    workspace = Dict{Symbol,Any}(
        :V=>V, :H=>H, :cs=>cs, :sn=>sn, :g=>g, :w=>w, :z=>z, :y=>y
    )

    # Helper mul! that always computes in the storage type of y
    mulA!(y, Aop, x) = mul!(y, Aop, x)

    # Initial residual in high precision
    mulA!(Ax, A, x)
    @inbounds @simd for i in 1:n
        r[i] = b[i] - Ax[i]
    end

    bnorm = max(norm(b), eps(T))
    rnorm = norm(r)
    history = T[rnorm]
    verbose && @info "[IR] iter=0  ‖r‖=$(rnorm)  rel=$(rnorm/bnorm)"

    # IR loop
    δ_lp = zeros(S, n)
    for k in 1:max_outer
        # Convert r to low precision RHS
        r_lp = S.(r)

        # Solve A_S δ ≈ r_S with restarted GMRES in low precision
        δ_lp .= 0
        # Loose inner tol: proportional to current residual, but not too tight (keeps S stable)
        inner_tol = max(1e-2, float(rtol))    # user-adjustable heuristic
        δ_lp, res_lp, it_in =
            gmres_lp!(δ_lp, A_lp, r_lp, Mop, workspace; restart=restart, tol=inner_tol, verbose=verbose)

        # Update x in high precision and recompute residual accurately
        @inbounds @simd for i in 1:n
            x[i] += T(δ_lp[i])
        end
        mulA!(Ax, A, x)
        @inbounds @simd for i in 1:n
            r[i] = b[i] - Ax[i]
        end

        rnorm = norm(r)
        push!(history, rnorm)
        verbose && @info "[IR] iter=$k  ‖r‖=$(rnorm)  rel=$(rnorm/bnorm)  (inner iters=$it_in)"

        # Convergence check
        if rnorm <= max(atol, rtol*bnorm)
            return x, k, history
        end
    end

    return x, max_outer, history
end

end # module


Main.MixedPrecisionGMRESIR

In [21]:
using LinearAlgebra
using .MixedPrecisionGMRESIR

n = size(A, 1)
# A = sprand(n, n, 0.01); A = A + A' + I # SPD-ish, but GMRES is general
x_true = randn(n)
b = Float64.(A * x_true)

x, iters, hist = MixedPrecisionGMRESIR.gmres_ir(A, b; S=Float32, restart=40, rtol=1e-8, max_outer=20, verbose=true)
println("rel err = ", norm(x - x_true)/norm(x_true))
println("outer iters = ", iters, ", final relres = ", last(hist)/norm(b))


[ Info: [IR] iter=0  ‖r‖=10332.259534796338  rel=1.0
[ Info:   [GMRES(S)]  iter=1  relres=0.7618659351065804
[ Info:   [GMRES(S)]  iter=2  relres=0.6765624335831899
[ Info:   [GMRES(S)]  iter=3  relres=0.554454799918755
[ Info:   [GMRES(S)]  iter=4  relres=0.50613665937873
[ Info:   [GMRES(S)]  iter=5  relres=0.42289195947642555
[ Info:   [GMRES(S)]  iter=6  relres=0.38652511854629984
[ Info:   [GMRES(S)]  iter=7  relres=0.3356298250519452
[ Info:   [GMRES(S)]  iter=8  relres=0.304160764785563
[ Info:   [GMRES(S)]  iter=9  relres=0.27141685424720685
[ Info:   [GMRES(S)]  iter=10  relres=0.24464879160193334
[ Info:   [GMRES(S)]  iter=11  relres=0.22186123746812084
[ Info:   [GMRES(S)]  iter=12  relres=0.19818199573333947
[ Info:   [GMRES(S)]  iter=13  relres=0.18282193519727877
[ Info:   [GMRES(S)]  iter=14  relres=0.1656220164971123
[ Info:   [GMRES(S)]  iter=15  relres=0.15227649244701336
[ Info:   [GMRES(S)]  iter=16  relres=0.1402258064502934
[ Info:   [GMRES(S)]  iter=17  relres=0.

rel err = 0.03901143090742542
outer iters = 20, final relres = 0.0004124338538339544


[ Info: [IR] iter=20  ‖r‖=4.261373618748674  rel=0.0004124338538339544  (inner iters=40)


## Iteration 2

In [22]:
module MixedPrecisionGMRESIR

using LinearAlgebra, SparseArrays

# -- Identity preconditioner
struct IdentityOp end
Base.size(::IdentityOp) = (0, 0)
function LinearAlgebra.mul!(y::AbstractVector, ::IdentityOp, x::AbstractVector)
    @inbounds @simd for i in eachindex(y, x)
        y[i] = x[i]
    end
    return y
end

# -- Low-precision restarted GMRES (left-preconditioned) --
# Solves approximately:  A_S * δ ≈ r_S   using GMRES(S)
# Returns δ in S, residual norm, and inner iters used.
function gmres_lp!(
    δ::AbstractVector{S},                # output solution (S)
    A,                                   # linear operator for A (mul!)
    r::AbstractVector{S},                # right-hand side (S)
    M,                                   # left preconditioner operator (mul!)
    workspace::Dict{Symbol,Any},         # preallocated buffers
    ; restart::Int=50, tol::Real=1e-4, maxiter::Int=0, verbose::Bool=false
) where {S<:AbstractFloat}

    n = length(r)
    maxiter = maxiter == 0 ? restart : maxiter

    # Buffers
    V  = workspace[:V]  :: Matrix{S}             # size (n, restart+1)
    H  = workspace[:H]  :: Matrix{Float64}       # size (restart+1, restart)
    cs = workspace[:cs] :: Vector{Float64}       # size (restart)
    sn = workspace[:sn] :: Vector{Float64}       # size (restart)
    g  = workspace[:g]  :: Vector{Float64}       # size (restart+1)
    w  = workspace[:w]  :: Vector{S}             # size (n)
    z  = workspace[:z]  :: Vector{S}             # size (n)

    # Initialize
    fill!(δ, zero(S))
    fill!(H, 0.0); fill!(cs, 0.0); fill!(sn, 0.0)
    fill!(w, zero(S)); fill!(z, zero(S))

    # v1 = M^{-1} r / β
    mul!(z, M, r)           # z = M \ r
    β = Float64(norm(z))
    if β == 0.0
        return δ, 0.0, 0
    end
    @inbounds @simd for i in 1:n
        V[i,1] = z[i] / S(β)
    end
    g .= 0.0
    g[1] = β

    jfinal = 0
    resnorm = β

    # Arnoldi with Givens rotations
    for j in 1:restart
        # w = M^{-1} (A * v_j)
        mul!(w, A, view(V, :, j))  # w = A*v_j
        mul!(z, M, w)              # z = M^{-1} * (A*v_j)
        copyto!(w, z)

        # Modified Gram-Schmidt
        for i in 1:j
            hij = Float64(dot(view(V, :, i), w))
            H[i, j] = hij
            @inbounds @simd for k in 1:n
                w[k] -= S(hij) * V[k, i]
            end
        end
        hj1 = Float64(norm(w))
        H[j+1, j] = hj1

        if hj1 == 0.0
            jfinal = j
            # happy breakdown
            break
        end

        @inbounds @simd for k in 1:n
            V[k, j+1] = w[k] / S(hj1)
        end

        # Apply previous Givens rotations to the new column
        for i in 1:j-1
            h_i  = H[i, j]
            h_ip = H[i+1, j]
            H[i, j]   =  cs[i]*h_i + sn[i]*h_ip
            H[i+1, j] = -sn[i]*h_i + cs[i]*h_ip
        end

        # Compute and apply j-th Givens
        r1 = H[j, j]
        r2 = H[j+1, j]
        denom = hypot(r1, r2)
        if denom == 0.0
            cs[j] = 1.0; sn[j] = 0.0
        else
            cs[j] = r1 / denom
            sn[j] = r2 / denom
        end
        H[j, j]   = cs[j]*r1 + sn[j]*r2
        H[j+1, j] = 0.0

        # Update g
        g_j  = g[j]
        g[j]   =  cs[j]*g_j
        g[j+1] = -sn[j]*g_j

        resnorm = abs(g[j+1])
        jfinal = j

        verbose && @info "  [GMRES(S)]  iter=$j  relres=$(resnorm/β)"

        if resnorm <= tol*β
            break
        end
    end

    # Backsolve for y (size jfinal)
    j = jfinal
    if j == 0
        return δ, resnorm, 0
    end
    y = workspace[:y]
    resize!(y, j)
    for i in j:-1:1
        s = g[i]
        for k in i+1:j
            s -= H[i,k] * y[k]
        end
        y[i] = s / H[i,i]
    end

    # δ = V[:,1:j] * y
    fill!(δ, zero(S))
    for k in 1:j
        α = S(y[k])
        @inbounds @simd for i in 1:n
            δ[i] += α * V[i, k]
        end
    end

    return δ, resnorm, j
end

# -- Public: Mixed-precision IR driver with adaptive inner tol --
function gmres_ir(
    A, b::AbstractVector{Tb};
    S::Type{<:AbstractFloat}=Float32,        # low precision
    restart::Int=100,                        # bigger default restart
    atol::Real=0.0,
    rtol::Real=1e-8,
    max_outer::Int=50,
    x0=nothing,
    M=nothing,
    verbose::Bool=false
) where {Tb<:AbstractFloat}

    n = length(b)
    T = Tb
    Mop = isnothing(M) ? IdentityOp() : M

    # high precision state
    x = isnothing(x0) ? zeros(T, n) : copy!(similar(b), T.(x0))
    r = similar(b)
    Ax = similar(b)

    # low-precision operator
    A_lp = A
    if A isa AbstractMatrix
        A_lp = convert(AbstractMatrix{S}, S.(A))
    end

    # workspace
    V  = zeros(S, n, restart+1)
    H  = zeros(Float64, restart+1, restart)
    cs = zeros(Float64, restart)
    sn = zeros(Float64, restart)
    g  = zeros(Float64, restart+1)
    w  = zeros(S, n)
    z  = zeros(S, n)
    y  = Float64[]
    workspace = Dict(:V=>V, :H=>H, :cs=>cs, :sn=>sn, :g=>g, :w=>w, :z=>z, :y=>y)

    mulA!(y, Aop, x) = mul!(y, Aop, x)

    # initial residual
    mulA!(Ax, A, x)
    @. r = b - Ax
    bnorm = max(norm(b), eps(T))
    rnorm = norm(r)
    history = T[rnorm]
    verbose && @info "[IR] iter=0  ‖r‖=$rnorm  rel=$(rnorm/bnorm)"

    δ_lp = zeros(S, n)
    for k in 1:max_outer
        r_lp = S.(r)

        # adaptive inner tolerance: scales with current outer residual
        inner_tol = min(0.5, 0.1 * (rnorm / bnorm))

        δ_lp .= 0
        δ_lp, res_lp, it_in =
            gmres_lp!(δ_lp, A_lp, r_lp, Mop, workspace;
                      restart=restart, tol=inner_tol, verbose=verbose)

        @. x += T(δ_lp)
        mulA!(Ax, A, x)
        @. r = b - Ax

        rnorm = norm(r)
        push!(history, rnorm)
        verbose && @info "[IR] iter=$k  ‖r‖=$rnorm  rel=$(rnorm/bnorm)  (inner iters=$it_in)"

        if rnorm <= max(atol, rtol*bnorm)
            return x, k, history
        end
    end

    return x, max_outer, history
end

end # module


Main.MixedPrecisionGMRESIR

In [23]:
using LinearAlgebra
using .MixedPrecisionGMRESIR

n = size(A, 1)
# A = sprand(n, n, 0.01); A = A + A' + I # SPD-ish, but GMRES is general
x_true = randn(n)
b = Float64.(A * x_true)

x, iters, hist = MixedPrecisionGMRESIR.gmres_ir(A, b; S=Float32, restart=40, rtol=1e-8, max_outer=20, verbose=true)
println("rel err = ", norm(x - x_true)/norm(x_true))
println("outer iters = ", iters, ", final relres = ", last(hist)/norm(b))


[ Info: [IR] iter=0  ‖r‖=10220.24126180786  rel=1.0


LoadError: MethodError: no method matching gmres_lp!(::Vector{Float32}, ::SparseArrays.SparseMatrixCSC{Float32, Int64}, ::Vector{Float32}, ::Main.MixedPrecisionGMRESIR.IdentityOp, ::Dict{Symbol, Array}; restart::Int64, tol::Float64, verbose::Bool)

[0mClosest candidates are:
[0m  gmres_lp!(::AbstractVector{S}, ::Any, ::AbstractVector{S}, ::Any, [91m::Dict{Symbol, Any}[39m; restart, tol, maxiter, verbose) where S<:AbstractFloat
[0m[90m   @[39m [35mMain.MixedPrecisionGMRESIR[39m [90m[4mIn[22]:18[24m[39m


## Iteration 3

In [24]:
module MixedPrecisionGMRESIR

using LinearAlgebra, SparseArrays

# ------------------------------
# Identity preconditioner
# ------------------------------
struct IdentityOp end
Base.size(::IdentityOp) = (0, 0)
function LinearAlgebra.mul!(y::AbstractVector, ::IdentityOp, x::AbstractVector)
    @inbounds @simd for i in eachindex(y, x)
        y[i] = x[i]
    end
    return y
end

# ------------------------------
# Low-precision restarted GMRES
# ------------------------------
function gmres_lp!(
    δ::AbstractVector{S},                # output solution (S)
    A,                                   # operator
    r::AbstractVector{S},                # RHS (S)
    M,                                   # left preconditioner
    workspace::Dict{Symbol,Any};
    restart::Int=50, tol::Real=1e-4, maxiter::Int=0, verbose::Bool=false
) where {S<:AbstractFloat}

    n = length(r)
    maxiter = maxiter == 0 ? restart : maxiter

    # Buffers
    V  = workspace[:V]  :: Matrix{S}
    H  = workspace[:H]  :: Matrix{Float64}
    cs = workspace[:cs] :: Vector{Float64}
    sn = workspace[:sn] :: Vector{Float64}
    g  = workspace[:g]  :: Vector{Float64}
    w  = workspace[:w]  :: Vector{S}
    z  = workspace[:z]  :: Vector{S}

    fill!(δ, zero(S))
    fill!(H, 0.0); fill!(cs, 0.0); fill!(sn, 0.0)
    fill!(w, zero(S)); fill!(z, zero(S))

    # v1 = M^{-1} r / β
    mul!(z, M, r)
    β = Float64(norm(z))
    if β == 0.0
        return δ, 0.0, 0
    end
    @inbounds @simd for i in 1:n
        V[i,1] = z[i] / S(β)
    end
    g .= 0.0
    g[1] = β

    jfinal = 0
    resnorm = β

    # Arnoldi with Givens rotations
    for j in 1:restart
        # w = M^{-1} * (A * v_j)
        mul!(w, A, view(V, :, j))
        mul!(z, M, w)
        copyto!(w, z)

        for i in 1:j
            hij = Float64(dot(view(V, :, i), w))
            H[i, j] = hij
            @inbounds @simd for k in 1:n
                w[k] -= S(hij) * V[k, i]
            end
        end
        hj1 = Float64(norm(w))
        H[j+1, j] = hj1

        if hj1 == 0.0
            jfinal = j
            break
        end

        @inbounds @simd for k in 1:n
            V[k, j+1] = w[k] / S(hj1)
        end

        # Apply previous Givens rotations
        for i in 1:j-1
            h_i, h_ip = H[i, j], H[i+1, j]
            H[i, j]   =  cs[i]*h_i + sn[i]*h_ip
            H[i+1, j] = -sn[i]*h_i + cs[i]*h_ip
        end

        # Compute/apply j-th Givens
        r1, r2 = H[j, j], H[j+1, j]
        denom = hypot(r1, r2)
        if denom == 0.0
            cs[j] = 1.0; sn[j] = 0.0
        else
            cs[j] = r1 / denom
            sn[j] = r2 / denom
        end
        H[j, j]   = cs[j]*r1 + sn[j]*r2
        H[j+1, j] = 0.0

        g_j = g[j]
        g[j]   =  cs[j]*g_j
        g[j+1] = -sn[j]*g_j

        resnorm = abs(g[j+1])
        jfinal = j

        verbose && @info "  [GMRES(S)] iter=$j  relres=$(resnorm/β)"

        if resnorm <= tol*β
            break
        end
    end

    # Backsolve for y
    j = jfinal
    if j == 0
        return δ, resnorm, 0
    end
    y = workspace[:y]
    resize!(y, j)
    for i in j:-1:1
        s = g[i]
        for k in i+1:j
            s -= H[i,k] * y[k]
        end
        y[i] = s / H[i,i]
    end

    # δ = V[:,1:j] * y
    fill!(δ, zero(S))
    for k in 1:j
        α = S(y[k])
        @inbounds @simd for i in 1:n
            δ[i] += α * V[i, k]
        end
    end

    return δ, resnorm, j
end

# ------------------------------
# Mixed-precision IR driver
# ------------------------------
function gmres_ir(
    A, b::AbstractVector{Tb};
    S::Type{<:AbstractFloat}=Float32,
    restart::Int=100,
    atol::Real=0.0,
    rtol::Real=1e-8,
    max_outer::Int=50,
    x0=nothing,
    M=nothing,
    verbose::Bool=false
) where {Tb<:AbstractFloat}

    n = length(b)
    T = Tb
    Mop = isnothing(M) ? IdentityOp() : M

    # high precision state
    x = isnothing(x0) ? zeros(T, n) : copy!(similar(b), T.(x0))
    r = similar(b)
    Ax = similar(b)

    # low-precision copy
    A_lp = A
    if A isa AbstractMatrix
        A_lp = convert(AbstractMatrix{S}, S.(A))
    end

    # workspace
    V  = zeros(S, n, restart+1)
    H  = zeros(Float64, restart+1, restart)
    cs = zeros(Float64, restart)
    sn = zeros(Float64, restart)
    g  = zeros(Float64, restart+1)
    w  = zeros(S, n)
    z  = zeros(S, n)
    y  = Float64[]
    workspace = Dict{Symbol,Any}(
        :V=>V, :H=>H, :cs=>cs, :sn=>sn, :g=>g, :w=>w, :z=>z, :y=>y
    )

    mulA!(y, Aop, x) = mul!(y, Aop, x)

    # initial residual
    mulA!(Ax, A, x)
    @. r = b - Ax
    bnorm = max(norm(b), eps(T))
    rnorm = norm(r)
    history = T[rnorm]
    verbose && @info "[IR] iter=0  ‖r‖=$rnorm  rel=$(rnorm/bnorm)"

    δ_lp = zeros(S, n)
    for k in 1:max_outer
        r_lp = S.(r)

        # adaptive inner tolerance
        inner_tol = min(0.5, 0.1 * (rnorm / bnorm))

        δ_lp .= 0
        δ_lp, res_lp, it_in =
            gmres_lp!(δ_lp, A_lp, r_lp, Mop, workspace;
                      restart=restart, tol=inner_tol, verbose=verbose)

        @. x += T(δ_lp)
        mulA!(Ax, A, x)
        @. r = b - Ax

        rnorm = norm(r)
        push!(history, rnorm)
        verbose && @info "[IR] iter=$k  ‖r‖=$rnorm  rel=$(rnorm/bnorm) (inner iters=$it_in)"

        if rnorm <= max(atol, rtol*bnorm)
            return x, k, history
        end
    end

    return x, max_outer, history
end

end # module


Main.MixedPrecisionGMRESIR

In [28]:
using LinearAlgebra
using .MixedPrecisionGMRESIR

n = size(A, 1)
# A = sprand(n, n, 0.01); A = A + A' + I # SPD-ish, but GMRES is general
x_true = randn(n)
b = Float64.(A * x_true)

x, iters, hist = MixedPrecisionGMRESIR.gmres_ir(A, b; S=Float32, restart=40, rtol=1e-8, max_outer=20, verbose=true)
println("rel err = ", norm(x - x_true)/norm(x_true))
println("outer iters = ", iters, ", final relres = ", last(hist)/norm(b))


[ Info: [IR] iter=0  ‖r‖=9892.07191684069  rel=1.0
[ Info:   [GMRES(S)] iter=1  relres=0.7255939629665591
[ Info:   [GMRES(S)] iter=2  relres=0.6374473931421838
[ Info:   [GMRES(S)] iter=3  relres=0.5246412754915564
[ Info:   [GMRES(S)] iter=4  relres=0.4840731320226117
[ Info:   [GMRES(S)] iter=5  relres=0.4067143822514403
[ Info:   [GMRES(S)] iter=6  relres=0.37064794970724524
[ Info:   [GMRES(S)] iter=7  relres=0.32851101386775655
[ Info:   [GMRES(S)] iter=8  relres=0.29703594612648204
[ Info:   [GMRES(S)] iter=9  relres=0.2635597419850298
[ Info:   [GMRES(S)] iter=10  relres=0.23811811666156876
[ Info:   [GMRES(S)] iter=11  relres=0.21865724461631805
[ Info:   [GMRES(S)] iter=12  relres=0.19793575926204
[ Info:   [GMRES(S)] iter=13  relres=0.18190407920362317
[ Info:   [GMRES(S)] iter=14  relres=0.16692914258822242
[ Info:   [GMRES(S)] iter=15  relres=0.15472533245706607
[ Info:   [GMRES(S)] iter=16  relres=0.1430154858163828
[ Info:   [GMRES(S)] iter=17  relres=0.13328193447261258

rel err = 0.0511097715475101
outer iters = 20, final relres = 0.0004406611410091557


[ Info: [IR] iter=20  ‖r‖=4.359051697819645  rel=0.0004406611410091557 (inner iters=40)


In [31]:
@btime MixedPrecisionGMRESIR.gmres_ir($A, $b; S=Float32, restart=40, rtol=1e-8, max_outer=20, verbose=false);
@btime A \ b;

  2.362 s (91784984 allocations: 1.38 GiB)
  181.025 ms (86 allocations: 151.94 MiB)


## Iteration 4

In [32]:
module MixedPrecisionGMRESIR

using LinearAlgebra, SparseArrays

struct IdentityOp end
(m::IdentityOp)(x) = x

"""
    gmres_lp!(x, A, b; restart=50, tol=0.25, maxiter=100, verbose=false)

Low-precision GMRES solve of A*x ≈ b.
Returns (x, niter, relres).
"""
function gmres_lp!(x::Vector{S}, A, b::Vector{S};
                   restart=50, tol=0.25, maxiter=100, verbose=false) where {S<:AbstractFloat}
    n = length(b)
    r = copy(b - A*x)
    β = norm(r)
    if β == 0
        return (x, 0, 0.0)
    end
    relres0 = 1.0

    iters = 0
    relres = relres0
    while relres > tol && iters < maxiter
        V = zeros(S, n, restart+1)
        H = zeros(S, restart+1, restart)
        g = zeros(S, restart+1)
        V[:,1] .= r / β
        g[1] = β

        k = 0
        for j = 1:restart
            w = A*V[:,j]
            for i=1:j
                H[i,j] = dot(V[:,i], w)
                w .-= H[i,j]*V[:,i]
            end
            H[j+1,j] = norm(w)
            if H[j+1,j] ≠ 0
                V[:,j+1] .= w / H[j+1,j]
            end

            y = H[1:j+1,1:j] \ g[1:j+1]
            rj = g[1:j+1] - H[1:j+1,1:j]*y
            relres = norm(rj)/β
            k = j
            if relres < tol
                break
            end
        end

        y = H[1:k+1,1:k] \ g[1:k+1]
        x .+= V[:,1:k]*y
        r = b - A*x
        β = norm(r)
        relres = β / norm(b)
        iters += k
        verbose && @info "[GMRES(S)] iter=$iters  relres=$relres"
    end
    return (x, iters, relres)
end

"""
    gmres_ir(A, b; S=Float32, restart=50, atol=1e-8, rtol=1e-6, max_outer=10, x0=nothing, verbose=false)

Mixed-precision iterative refinement with GMRES(S) inner solves.
"""
function gmres_ir(A::SparseMatrixCSC{Float64}, b::Vector{Float64};
                  S=Float32, restart=50, atol=1e-8, rtol=1e-6,
                  max_outer=10, x0=nothing, verbose=false)

    n = length(b)
    x = x0 === nothing ? zeros(Float64,n) : copy(x0)
    r = b - A*x
    rnorm0 = norm(r)
    bnorm  = norm(b)
    verbose && @info "[IR] iter=0  ‖r‖=$rnorm0  rel=1.0"

    for outer = 1:max_outer
        if norm(r) <= atol + rtol*bnorm
            return (x, outer-1, norm(r)/bnorm)
        end
        # Solve correction system in S
        rs = convert.(S, r)
        d = zeros(S, n)
        d, niters, relres = gmres_lp!(d, convert(S,A), rs; restart=restart, tol=0.25, verbose=verbose)

        # Update in high precision
        x .+= Float64.(d)
        r = b - A*x
        rel = norm(r)/rnorm0
        verbose && @info "[IR] iter=$outer  ‖r‖=$(norm(r))  rel=$rel (inner iters=$niters)"
    end
    return (x, max_outer, norm(r)/bnorm)
end

end # module


Main.MixedPrecisionGMRESIR

In [33]:
using LinearAlgebra
using .MixedPrecisionGMRESIR

n = size(A, 1)
# A = sprand(n, n, 0.01); A = A + A' + I # SPD-ish, but GMRES is general
x_true = randn(n)
b = Float64.(A * x_true)

x, iters, hist = MixedPrecisionGMRESIR.gmres_ir(A, b; S=Float32, restart=40, rtol=1e-8, max_outer=20, verbose=true)
println("rel err = ", norm(x - x_true)/norm(x_true))
println("outer iters = ", iters, ", final relres = ", last(hist)/norm(b))

[ Info: [IR] iter=0  ‖r‖=10358.307689376425  rel=1.0


LoadError: MethodError: [0mCannot `convert` an object of type [92mSparseArrays.SparseMatrixCSC{Float64, Int64}[39m[0m to an object of type [91mFloat32[39m

[0mClosest candidates are:
[0m  convert(::Type{T}, [91m::T[39m) where T<:Number
[0m[90m   @[39m [90mBase[39m [90m[4mnumber.jl:6[24m[39m
[0m  convert(::Type{T}, [91m::T[39m) where T
[0m[90m   @[39m [90mBase[39m [90m[4mBase.jl:84[24m[39m
[0m  convert(::Type{T}, [91m::AbstractChar[39m) where T<:Number
[0m[90m   @[39m [90mBase[39m [90m[4mchar.jl:185[24m[39m
[0m  ...


## Iteration 5

In [34]:
module MixedPrecisionGMRESIR

using LinearAlgebra
using SparseArrays
using Printf

# Identity preconditioner
struct IdentityOp end
(m::IdentityOp)(x) = x

"""
    gmres_lp!(x, A, b; restart, tol, maxiter, verbose)

Low-precision restarted GMRES solver.
Solves A*x ≈ b starting from x=0, with restart length and tolerance.
"""
function gmres_lp!(x::AbstractVector{S}, A, b::AbstractVector{S};
                   restart::Int=40, tol::Float64=1e-2,
                   maxiter::Int=10^4, verbose::Bool=false) where {S<:AbstractFloat}

    n = length(b)
    r = copy(b)
    beta = norm(r)
    if beta == 0
        fill!(x, 0)
        return x, 0
    end

    V = Matrix{S}(undef, n, restart+1)
    H = Matrix{S}(undef, restart+1, restart)
    fill!(x, 0)

    outer_iter = 0
    while beta > tol*norm(b) && outer_iter*restart < maxiter
        V[:,1] .= r / beta
        g = zeros(S, restart+1)
        g[1] = beta

        for j in 1:restart
            w = A * V[:,j]
            for i in 1:j
                H[i,j] = dot(V[:,i], w)
                w -= H[i,j] * V[:,i]
            end
            H[j+1,j] = norm(w)
            if H[j+1,j] ≈ 0
                break
            end
            V[:,j+1] = w / H[j+1,j]

            # solve least squares
            y = H[1:j+1,1:j] \ g[1:j+1]
            x .= V[:,1:j] * y
            r = b - A*x
            beta = norm(r)

            verbose && @info "  [GMRES(S)] iter=$j  relres=$(beta/norm(b))"

            if beta <= tol*norm(b)
                return x, outer_iter*restart + j
            end
        end
        outer_iter += 1
    end
    return x, outer_iter*restart
end


"""
    gmres_ir(A, b; S=Float32, restart=40, atol=1e-6, rtol=1e-12,
             max_outer=50, x0=nothing, verbose=true)

Mixed-precision iterative refinement with low-precision GMRES.
"""
function gmres_ir(A::SparseMatrixCSC{Float64,Int}, b::Vector{Float64};
                  S::Type{<:AbstractFloat}=Float32,
                  restart::Int=40, atol::Float64=1e-6, rtol::Float64=1e-12,
                  max_outer::Int=50, x0=nothing, verbose::Bool=true)

    # Low-precision versions
    A_S = sparse(A.rowval, A.colptr, convert.(S, A.nzval), size(A,1), size(A,2))
    b_S = convert.(S, b)

    n = length(b)
    x = x0 === nothing ? zeros(Float64, n) : copy(x0)

    r = b - A*x
    bnorm = norm(b)
    relres = norm(r) / bnorm

    verbose && @info "[IR] iter=0  ‖r‖=$(norm(r))  rel=1.0"

    for outer in 1:max_outer
        if relres < max(rtol, atol/bnorm)
            break
        end

        # Cast residual to low precision
        r_S = convert.(S, r)
        d_S = zeros(S, n)

        d_S, iters = gmres_lp!(d_S, A_S, r_S; restart=restart, tol=0.1, verbose=verbose)

        # Correct in high precision
        x .+= convert.(Float64, d_S)

        # New residual in high precision
        r = b - A*x
        relres = norm(r) / bnorm

        verbose && @info "[IR] iter=$outer  ‖r‖=$(norm(r))  rel=$relres (inner iters=$iters)"
    end

    return x, relres
end

end # module


Main.MixedPrecisionGMRESIR

In [35]:
using LinearAlgebra
using .MixedPrecisionGMRESIR

n = size(A, 1)
# A = sprand(n, n, 0.01); A = A + A' + I # SPD-ish, but GMRES is general
x_true = randn(n)
b = Float64.(A * x_true)

x, iters, hist = MixedPrecisionGMRESIR.gmres_ir(A, b; S=Float32, restart=40, rtol=1e-8, max_outer=20, verbose=true)
println("rel err = ", norm(x - x_true)/norm(x_true))
println("outer iters = ", iters, ", final relres = ", last(hist)/norm(b))

LoadError: ArgumentError: the first three arguments' lengths must match, length(I) (=484256) == length(J) (= 23561) == length(V) (= 484256)

## Iteration 6

In [36]:
module MixedPrecisionGMRESIR

using LinearAlgebra
using SparseArrays
using Printf

# Identity preconditioner
struct IdentityOp end
(m::IdentityOp)(x) = x

"""
    gmres_lp!(x, A, b; restart, tol, maxiter, verbose)

Low-precision restarted GMRES solver.
Solves A*x ≈ b starting from x=0, with restart length and tolerance.
"""
function gmres_lp!(x::AbstractVector{S}, A, b::AbstractVector{S};
                   restart::Int=40, tol::Float64=1e-2,
                   maxiter::Int=10^4, verbose::Bool=false) where {S<:AbstractFloat}

    n = length(b)
    r = copy(b)
    beta = norm(r)
    if beta == 0
        fill!(x, 0)
        return x, 0
    end

    V = Matrix{S}(undef, n, restart+1)
    H = Matrix{S}(undef, restart+1, restart)
    fill!(x, 0)

    outer_iter = 0
    while beta > tol*norm(b) && outer_iter*restart < maxiter
        V[:,1] .= r / beta
        g = zeros(S, restart+1)
        g[1] = beta

        for j in 1:restart
            w = A * V[:,j]
            for i in 1:j
                H[i,j] = dot(V[:,i], w)
                w -= H[i,j] * V[:,i]
            end
            H[j+1,j] = norm(w)
            if H[j+1,j] ≈ 0
                break
            end
            V[:,j+1] = w / H[j+1,j]

            # solve least squares
            y = H[1:j+1,1:j] \ g[1:j+1]
            x .= V[:,1:j] * y
            r = b - A*x
            beta = norm(r)

            verbose && @info "  [GMRES(S)] iter=$j  relres=$(beta/norm(b))"

            if beta <= tol*norm(b)
                return x, outer_iter*restart + j
            end
        end
        outer_iter += 1
    end
    return x, outer_iter*restart
end


"""
    gmres_ir(A, b; S=Float32, restart=40, atol=1e-6, rtol=1e-12,
             max_outer=50, x0=nothing, verbose=true)

Mixed-precision iterative refinement with low-precision GMRES.
"""
function gmres_ir(A::SparseMatrixCSC{Float64,Int}, b::Vector{Float64};
                  S::Type{<:AbstractFloat}=Float32,
                  restart::Int=40, atol::Float64=1e-6, rtol::Float64=1e-12,
                  max_outer::Int=50, x0=nothing, verbose::Bool=true)

    # Proper low-precision conversion of sparse matrix
    A_S = SparseMatrixCSC(A.m, A.n, A.colptr, A.rowval, convert.(S, A.nzval))
    b_S = convert.(S, b)

    n = length(b)
    x = x0 === nothing ? zeros(Float64, n) : copy(x0)

    r = b - A*x
    bnorm = norm(b)
    relres = norm(r) / bnorm

    verbose && @info "[IR] iter=0  ‖r‖=$(norm(r))  rel=1.0"

    for outer in 1:max_outer
        if relres < max(rtol, atol/bnorm)
            break
        end

        # Cast residual to low precision
        r_S = convert.(S, r)
        d_S = zeros(S, n)

        d_S, iters = gmres_lp!(d_S, A_S, r_S; restart=restart, tol=0.1, verbose=verbose)

        # Correct in high precision
        x .+= convert.(Float64, d_S)

        # New residual in high precision
        r = b - A*x
        relres = norm(r) / bnorm

        verbose && @info "[IR] iter=$outer  ‖r‖=$(norm(r))  rel=$relres (inner iters=$iters)"
    end

    return x, relres
end

end # module


Main.MixedPrecisionGMRESIR

In [37]:
using LinearAlgebra
using .MixedPrecisionGMRESIR

n = size(A, 1)
# A = sprand(n, n, 0.01); A = A + A' + I # SPD-ish, but GMRES is general
x_true = randn(n)
b = Float64.(A * x_true)

x, iters, hist = MixedPrecisionGMRESIR.gmres_ir(A, b; S=Float32, restart=40, rtol=1e-8, max_outer=20, verbose=true)
println("rel err = ", norm(x - x_true)/norm(x_true))
println("outer iters = ", iters, ", final relres = ", last(hist)/norm(b))

[ Info: [IR] iter=0  ‖r‖=10297.968397017947  rel=1.0
[ Info:   [GMRES(S)] iter=1  relres=0.771251
[ Info:   [GMRES(S)] iter=2  relres=0.688833
[ Info:   [GMRES(S)] iter=3  relres=0.55445576
[ Info:   [GMRES(S)] iter=4  relres=0.49846607
[ Info:   [GMRES(S)] iter=5  relres=0.4220683
[ Info:   [GMRES(S)] iter=6  relres=0.38537705
[ Info:   [GMRES(S)] iter=7  relres=NaN
[ Info:   [GMRES(S)] iter=8  relres=NaN
[ Info:   [GMRES(S)] iter=9  relres=NaN
[ Info:   [GMRES(S)] iter=10  relres=NaN
[ Info:   [GMRES(S)] iter=11  relres=NaN
[ Info:   [GMRES(S)] iter=12  relres=NaN
[ Info:   [GMRES(S)] iter=13  relres=NaN
[ Info:   [GMRES(S)] iter=14  relres=NaN
[ Info:   [GMRES(S)] iter=15  relres=NaN
[ Info:   [GMRES(S)] iter=16  relres=NaN
[ Info:   [GMRES(S)] iter=17  relres=NaN
[ Info:   [GMRES(S)] iter=18  relres=NaN
[ Info:   [GMRES(S)] iter=19  relres=NaN
[ Info:   [GMRES(S)] iter=20  relres=NaN
[ Info:   [GMRES(S)] iter=21  relres=NaN
[ Info:   [GMRES(S)] iter=22  relres=NaN
[ Info:   [GMRES(

LoadError: BoundsError: attempt to access Tuple{Vector{Float64}, Float64} at index [3]

## Iteration 7

In [38]:
module MixedPrecisionGMRESIR

using LinearAlgebra
using SparseArrays
using Printf
using Logging

# --------------------------
# Utilities
# --------------------------

# Safe dot that keeps result in element type S.
@inline function dotS(u::AbstractVector{S}, v::AbstractVector{S}) where {S<:AbstractFloat}
    s = zero(S)
    @inbounds @simd for i in eachindex(u,v)
        s = muladd(u[i], v[i], s)
    end
    return s
end

# Apply and update Givens rotations to the column j of H, and update residual vector g.
@inline function apply_givens!(H::AbstractMatrix{S}, g::AbstractVector{S}, cs::AbstractVector{S}, sn::AbstractVector{S}, j::Int) where {S<:AbstractFloat}
    # Apply previous rotations
    @inbounds for i = 1:j-1
        h1 = H[i, j]
        h2 = H[i+1, j]
        H[i,   j] =  cs[i]*h1 + sn[i]*h2
        H[i+1, j] = -sn[i]*h1 + cs[i]*h2
    end
    # Create new rotation to zero H[j+1, j]
    h1 = H[j, j]
    h2 = H[j+1, j]
    denom = hypot(h1, h2) # computed in S
    if denom == 0
        cs[j] = one(S)
        sn[j] = zero(S)
    else
        cs[j] = h1 / denom
        sn[j] = h2 / denom
    end
    # Apply it
    H[j,   j] = cs[j]*h1 + sn[j]*h2
    H[j+1, j] = zero(S)
    # Update g (same rotation)
    g_j   = g[j]
    g_j1  = g[j+1]
    g[j]   =  cs[j]*g_j + sn[j]*g_j1
    g[j+1] = -sn[j]*g_j + cs[j]*g_j1
    return nothing
end

# Back-substitution for small upper-triangular R y = g
@inline function backsolve_upper!(y::AbstractVector{S}, R::AbstractMatrix{S}, g::AbstractVector{S}, j::Int) where {S<:AbstractFloat}
    @inbounds for k = j:-1:1
        s = g[k]
        for i = k+1:j
            s -= R[k,i]*y[i]
        end
        y[k] = s / R[k,k]
    end
    return y
end

# --------------------------
# Low-precision GMRES (restartable), left-preconditioned M^{-1}A
# --------------------------

"""
    gmres_lp!(z, A, r; restart=40, tol=1e-2, maxiter=10_000, verbose=false, M=nothing, reorth=true)

Compute an approximate solution `z` to `M^{-1}A z ≈ M^{-1} r` using **restarted GMRES** in low precision.
- All arrays must be of element type `S <: AbstractFloat` (typically `Float32` or `Float16` if supported).
- `M` can be a left-preconditioner; pass a callable `M(y) = M^{-1} y`. Use `nothing` for identity.
- Robust implementation with Givens rotations and happy-breakdown handling (no NaNs).
Returns `(z, iters_done::Int, relres::Float64)`.
"""
function gmres_lp!(z::AbstractVector{S}, A::Union{SparseMatrixCSC{S},AbstractMatrix{S}}, r::AbstractVector{S};
                   restart::Int=40, tol::Float64=1e-2, maxiter::Int=10_000, verbose::Bool=false,
                   M=nothing, reorth::Bool=true) where {S<:AbstractFloat}

    n = length(r)
    fill!(z, zero(S))

    # storage
    V = Matrix{S}(undef, n, restart+1)
    H = zeros(S, restart+1, restart)
    cs = zeros(S, restart)
    sn = zeros(S, restart)
    g  = zeros(S, restart+1)
    y  = zeros(S, restart)

    # define preconditioner
    M⁻¹ = M === nothing ? (y->y) : M

    # initial residual in low precision space: M^{-1} r
    v0 = M⁻¹(r)
    β = norm(v0)
    bnorm = β
    if β == 0
        return z, 0, 0.0
    end

    V[:,1] .= v0 ./ β
    total_iters = 0

    while total_iters < maxiter
        fill!(H, zero(S))
        fill!(cs, zero(S))
        fill!(sn, zero(S))
        fill!(g,  zero(S))
        fill!(y,  zero(S))
        g[1] = β

        jstop = 0
        for j = 1:restart
            # Arnoldi: w = M^{-1}A * V[:,j]
            w = M⁻¹(A * V[:,j])

            # Modified Gram-Schmidt
            @inbounds for i = 1:j
                H[i,j] = dotS(V[:,i], w)
                w .-= H[i,j] .* V[:,i]
            end
            if reorth
                # One step of re-orthogonalization for stability
                @inbounds for i = 1:j
                    hcor = dotS(V[:,i], w)
                    H[i,j] += hcor
                    w .-= hcor .* V[:,i]
                end
            end

            H[j+1, j] = norm(w)

            if H[j+1,j] == 0  # happy breakdown
                # Apply and create rotations up to j using zeros at position j+1,j
                apply_givens!(H, g, cs, sn, j)
                jstop = j
                total_iters += 1
                break
            else
                # Normal step
                V[:, j+1] .= w ./ H[j+1, j]
                apply_givens!(H, g, cs, sn, j)
                total_iters += 1
            end

            relres_now = Float64(abs(g[j+1])) / Float64(bnorm)
            verbose && @info "  [GMRES(S)] iter=$j  relres=$(round(relres_now, sigdigits=7))"

            if relres_now <= tol
                jstop = j
                break
            end
        end

        jstop == 0 && (jstop = restart)

        # Solve least squares via back-substitution on R (upper jstop×jstop)
        backsolve_upper!(y, view(H, 1:jstop, 1:jstop), view(g, 1:jstop), jstop)

        # Accumulate solution increment (in S)
        z .+= V[:, 1:jstop] * y

        # Compute new residual for possible next restart
        vres = M⁻¹(r - A*z)  # still in S
        β = norm(vres)
        bnorm0 = bnorm == 0 ? one(S) : bnorm
        relres = Float64(β) / Float64(bnorm0)

        if relres <= tol || total_iters >= maxiter
            return z, total_iters, relres
        end

        # Restart: set first Krylov vector to normalized residual
        V[:,1] .= vres ./ β
        # Continue
    end

    return z, total_iters, Float64(abs(g[end]))/Float64(bnorm)
end

# --------------------------
# Iterative Refinement (outer, high precision)
# --------------------------

"""
    gmres_ir(A, b; S=Float32, restart=40, atol=1e-12, rtol=1e-12,
             max_outer=50, x0=nothing, verbose=true, jacobi=false)

Mixed-precision **iterative refinement** that uses a low-precision, restarted **GMRES** inner solver.

- High precision: `Float64` (outer residuals and updates).
- Low precision: `S` (typically `Float32`) for inner GMRES and matvecs.

Keyword arguments:
- `restart`: GMRES restart length.
- `atol`, `rtol`: stopping criteria on high-precision residual (`‖r‖ <= max(rtol‖b‖, atol)`).
- `max_outer`: maximum outer IR iterations.
- `x0`: optional initial guess (Float64).
- `verbose`: print progress.
- `jacobi`: if `true`, use a **left Jacobi preconditioner** in low precision.

Returns `(x::Vector{Float64}, relres::Float64, outer_iters::Int, inner_total_iters::Int)`.
"""
function gmres_ir(A::SparseMatrixCSC{Float64,Int}, b::Vector{Float64};
                  S::Type{<:AbstractFloat}=Float32,
                  restart::Int=40, atol::Float64=1e-12, rtol::Float64=1e-12,
                  max_outer::Int=50, x0=nothing, verbose::Bool=true, jacobi::Bool=false)

    # Low-precision operator: same sparsity, converted values
    A_S = SparseMatrixCSC(A.m, A.n, A.colptr, A.rowval, convert.(S, A.nzval))

    # Optional left Jacobi preconditioner in low precision
    M = nothing
    if jacobi
        d = Vector{S}(undef, size(A_S,1))
        @inbounds for i in 1:length(d)
            di = S(A_S[i,i])
            d[i] = (di == 0) ? one(S) : inv(di)
        end
        M = (y::AbstractVector{S}) -> (d .* y)
    end

    n = length(b)
    x = x0 === nothing ? zeros(Float64, n) : copy(x0)

    bnorm = norm(b)
    r = b - A*x
    rnorm = norm(r)
    relres = rnorm / (bnorm == 0 ? 1.0 : bnorm)
    verbose && @info "[IR] iter=0  ‖r‖=$(rnorm)  rel=1.0"

    if relres <= max(rtol, atol/(bnorm == 0 ? 1.0 : bnorm))
        return x, relres, 0, 0
    end

    inner_total = 0

    for outer = 1:max_outer
        # Adaptive inner tolerance (looser at start, tighter as we proceed)
        # A practical heuristic: eta ∈ [1e-3, 1e-1], proportional to current outer relres.
        eta = clamp(0.2 * relres, 1e-3, 1e-1)

        # Low-precision residual
        r_S = convert.(S, r)

        # Solve for low-precision correction d_S in M^{-1} A d ≈ M^{-1} r
        d_S = zeros(S, n)
        d_S, iters_in, relres_in = gmres_lp!(d_S, A_S, r_S; restart=restart, tol=eta, verbose=verbose, M=M, reorth=true)
        inner_total += iters_in

        # High-precision correction
        x .+= convert.(Float64, d_S)

        # New high-precision residual
        r = b - A*x
        rnorm = norm(r)
        relres = rnorm / (bnorm == 0 ? 1.0 : bnorm)
        verbose && @info "[IR] iter=$(outer)  ‖r‖=$(rnorm)  rel=$(relres)  (inner iters=$(iters_in))"

        if relres <= max(rtol, atol/(bnorm == 0 ? 1.0 : bnorm))
            return x, relres, outer, inner_total
        end
    end

    return x, relres, max_outer, inner_total
end

# --------------------------
# Simple driver (example)
# --------------------------

"""
    solve_with_ir(A, b; S=Float32, restart=40, rtol=1e-12, atol=1e-12, max_outer=50, verbose=true, jacobi=false)

Convenience wrapper that calls `gmres_ir` and prints a short summary.
Returns `x`.
"""
function solve_with_ir(A::SparseMatrixCSC{Float64,Int}, b::Vector{Float64};
                       S::Type{<:AbstractFloat}=Float32, restart::Int=40,
                       rtol::Float64=1e-12, atol::Float64=1e-12,
                       max_outer::Int=50, verbose::Bool=true, jacobi::Bool=false)

    x, relres, outer, inner_total = gmres_ir(A, b; S=S, restart=restart,
                                             atol=atol, rtol=rtol,
                                             max_outer=max_outer, verbose=verbose, jacobi=jacobi)
    verbose && @info "DONE: outer=$outer, inner_total=$inner_total, final relres=$(relres)"
    return x
end

end # module


LoadError: syntax: local variable S cannot be used in closure declaration

## Iteration 8

In [39]:
module MixedPrecisionGMRESIR

using LinearAlgebra
using SparseArrays
using Printf
using Logging

# --------------------------
# Utilities
# --------------------------

# Safe dot that keeps result in element type S.
@inline function dotS(u::AbstractVector{S}, v::AbstractVector{S}) where {S<:AbstractFloat}
    s = zero(S)
    @inbounds @simd for i in eachindex(u,v)
        s = muladd(u[i], v[i], s)
    end
    return s
end

# Apply and update Givens rotations to the column j of H, and update residual vector g.
@inline function apply_givens!(H::AbstractMatrix{S}, g::AbstractVector{S}, cs::AbstractVector{S}, sn::AbstractVector{S}, j::Int) where {S<:AbstractFloat}
    # Apply previous rotations
    @inbounds for i = 1:j-1
        h1 = H[i, j]
        h2 = H[i+1, j]
        H[i,   j] =  cs[i]*h1 + sn[i]*h2
        H[i+1, j] = -sn[i]*h1 + cs[i]*h2
    end
    # Create new rotation to zero H[j+1, j]
    h1 = H[j, j]
    h2 = H[j+1, j]
    denom = hypot(h1, h2) # computed in S
    if denom == 0
        cs[j] = one(S)
        sn[j] = zero(S)
    else
        cs[j] = h1 / denom
        sn[j] = h2 / denom
    end
    # Apply it
    H[j,   j] = cs[j]*h1 + sn[j]*h2
    H[j+1, j] = zero(S)
    # Update g (same rotation)
    g_j   = g[j]
    g_j1  = g[j+1]
    g[j]   =  cs[j]*g_j + sn[j]*g_j1
    g[j+1] = -sn[j]*g_j + cs[j]*g_j1
    return nothing
end

# Back-substitution for small upper-triangular R y = g (robust to zero pivot)
@inline function backsolve_upper!(y::AbstractVector{S}, R::AbstractMatrix{S}, g::AbstractVector{S}, j::Int) where {S<:AbstractFloat}
    @inbounds for k = j:-1:1
        s = g[k]
        for i = k+1:j
            s -= R[k,i]*y[i]
        end
        rk = R[k,k]
        if rk == 0
            y[k] = zero(S)  # consistent minimum-norm if breakdown produced a zero
        else
            y[k] = s / rk
        end
    end
    return y
end

# --------------------------
# Low-precision GMRES (restartable), left-preconditioned M^{-1}A
# --------------------------

"""
    gmres_lp!(z, A, r; restart=40, tol=1e-2, maxiter=10_000, verbose=false, M=nothing, reorth=true)

Compute an approximate solution `z` to `M^{-1}A z ≈ M^{-1} r` using **restarted GMRES** in low precision.
- All arrays must be of element type `S <: AbstractFloat` (typically `Float32`).
- `M` can be a left-preconditioner; pass a callable `M(y) = M^{-1} y`. Use `nothing` for identity.
Returns `(z, iters_done::Int, relres::Float64)`.
"""
function gmres_lp!(z::AbstractVector{S}, A::Union{SparseMatrixCSC{S},AbstractMatrix{S}}, r::AbstractVector{S};
                   restart::Int=40, tol::Float64=1e-2, maxiter::Int=10_000, verbose::Bool=false,
                   M=nothing, reorth::Bool=true) where {S<:AbstractFloat}

    n = length(r)
    fill!(z, zero(S))

    # storage
    V = Matrix{S}(undef, n, restart+1)
    H = zeros(S, restart+1, restart)
    cs = zeros(S, restart)
    sn = zeros(S, restart)
    g  = zeros(S, restart+1)
    y  = zeros(S, restart)

    # define preconditioner
    M⁻¹ = M === nothing ? (y->y) : M

    # initial residual in low precision space: M^{-1} r
    v0 = M⁻¹(r)
    β = norm(v0)
    bnorm = β
    if β == 0
        return z, 0, 0.0
    end

    V[:,1] .= v0 ./ β
    total_iters = 0

    while total_iters < maxiter
        fill!(H, zero(S))
        fill!(cs, zero(S))
        fill!(sn, zero(S))
        fill!(g,  zero(S))
        fill!(y,  zero(S))
        g[1] = β

        jstop = 0
        for j = 1:restart
            # Arnoldi: w = M^{-1}A * V[:,j]
            w = M⁻¹(A * V[:,j])

            # Modified Gram-Schmidt
            @inbounds for i = 1:j
                H[i,j] = dotS(V[:,i], w)
                w .-= H[i,j] .* V[:,i]
            end
            if reorth
                # One step of re-orthogonalization for stability
                @inbounds for i = 1:j
                    hcor = dotS(V[:,i], w)
                    H[i,j] += hcor
                    w .-= hcor .* V[:,i]
                end
            end

            H[j+1, j] = norm(w)

            if H[j+1,j] == 0  # happy breakdown
                apply_givens!(H, g, cs, sn, j)
                jstop = j
                total_iters += 1
                break
            else
                V[:, j+1] .= w ./ H[j+1, j]
                apply_givens!(H, g, cs, sn, j)
                total_iters += 1
            end

            relres_now = Float64(abs(g[j+1])) / Float64(bnorm)
            verbose && @info "  [GMRES(S)] iter=$j  relres=$(round(relres_now, sigdigits=7))"

            if relres_now <= tol
                jstop = j
                break
            end
        end

        jstop == 0 && (jstop = restart)

        # Solve least squares via back-substitution on R (upper jstop×jstop)
        backsolve_upper!(y, view(H, 1:jstop, 1:jstop), view(g, 1:jstop), jstop)

        # Accumulate solution increment (in S)
        z .+= V[:, 1:jstop] * y

        # Compute new residual for possible next restart
        vres = M⁻¹(r - A*z)  # still in S
        β = norm(vres)
        bnorm0 = bnorm == 0 ? one(S) : bnorm
        relres = Float64(β) / Float64(bnorm0)

        if relres <= tol || total_iters >= maxiter
            return z, total_iters, relres
        end

        # Restart: set first Krylov vector to normalized residual
        V[:,1] .= vres ./ β
    end

    return z, total_iters, Float64(abs(g[end]))/Float64(bnorm)
end

# --------------------------
# Iterative Refinement (outer, high precision)
# --------------------------

"""
    gmres_ir(A, b; S=Float32, restart=40, atol=1e-12, rtol=1e-12,
             max_outer=50, x0=nothing, verbose=true, jacobi=false)

Mixed-precision **iterative refinement** that uses a low-precision, restarted **GMRES** inner solver.

- High precision: `Float64` (outer residuals and updates).
- Low precision: `S` (typically `Float32`) for inner GMRES and matvecs.

Keyword arguments:
- `restart`: GMRES restart length.
- `atol`, `rtol`: stopping criteria on high-precision residual (`‖r‖ <= max(rtol‖b‖, atol)`).
- `max_outer`: maximum outer IR iterations.
- `x0`: optional initial guess (Float64).
- `verbose`: print progress.
- `jacobi`: if `true`, use a **left Jacobi preconditioner** in low precision.

Returns `(x::Vector{Float64}, relres::Float64, outer_iters::Int, inner_total_iters::Int)`.
"""
function gmres_ir(A::SparseMatrixCSC{Float64,Int}, b::Vector{Float64};
                  S::Type{<:AbstractFloat}=Float32,
                  restart::Int=40, atol::Float64=1e-12, rtol::Float64=1e-12,
                  max_outer::Int=50, x0=nothing, verbose::Bool=true, jacobi::Bool=false)

    # Low-precision operator: same sparsity, converted values (share structure)
    A_S = SparseMatrixCSC(A.m, A.n, A.colptr, A.rowval, convert.(S, A.nzval))

    # Optional left Jacobi preconditioner in low precision
    M = nothing
    if jacobi
        d = Vector{S}(undef, size(A_S,1))
        @inbounds for i in 1:length(d)
            di = S(A_S[i,i])
            d[i] = (di == 0) ? one(S) : inv(di)
        end
        # NOTE: no type annotation here; using S in a closure annotation triggers the error you hit.
        M = (y) -> d .* y
    end

    n = length(b)
    x = x0 === nothing ? zeros(Float64, n) : copy(x0)

    bnorm = norm(b)
    r = b - A*x
    rnorm = norm(r)
    relres = rnorm / (bnorm == 0 ? 1.0 : bnorm)
    verbose && @info "[IR] iter=0  ‖r‖=$(rnorm)  rel=1.0"

    if relres <= max(rtol, atol/(bnorm == 0 ? 1.0 : bnorm))
        return x, relres, 0, 0
    end

    inner_total = 0

    for outer = 1:max_outer
        # Adaptive inner tolerance (looser at start, tighter as we proceed)
        eta = clamp(0.2 * relres, 1e-3, 1e-1)

        # Low-precision residual
        r_S = convert.(S, r)

        # Solve for low-precision correction in M^{-1} A d ≈ M^{-1} r
        d_S = zeros(S, n)
        d_S, iters_in, relres_in = gmres_lp!(d_S, A_S, r_S; restart=restart, tol=eta, verbose=verbose, M=M, reorth=true)
        inner_total += iters_in

        # High-precision correction
        x .+= convert.(Float64, d_S)

        # New high-precision residual
        r = b - A*x
        rnorm = norm(r)
        relres = rnorm / (bnorm == 0 ? 1.0 : bnorm)
        verbose && @info "[IR] iter=$(outer)  ‖r‖=$(rnorm)  rel=$(relres)  (inner iters=$(iters_in))"

        if relres <= max(rtol, atol/(bnorm == 0 ? 1.0 : bnorm))
            return x, relres, outer, inner_total
        end
    end

    return x, relres, max_outer, inner_total
end

# --------------------------
# Simple driver (example)
# --------------------------

"""
    solve_with_ir(A, b; S=Float32, restart=40, rtol=1e-12, atol=1e-12, max_outer=50, verbose=true, jacobi=false)

Convenience wrapper that calls `gmres_ir` and prints a short summary.
Returns `x`.
"""
function solve_with_ir(A::SparseMatrixCSC{Float64,Int}, b::Vector{Float64};
                       S::Type{<:AbstractFloat}=Float32, restart::Int=40,
                       rtol::Float64=1e-12, atol::Float64=1e-12,
                       max_outer::Int=50, verbose::Bool=true, jacobi::Bool=false)

    x, relres, outer, inner_total = gmres_ir(A, b; S=S, restart=restart,
                                             atol=atol, rtol=rtol,
                                             max_outer=max_outer, verbose=verbose, jacobi=jacobi)
    verbose && @info "DONE: outer=$outer, inner_total=$inner_total, final relres=$(relres)"
    return x
end

end # module

Main.MixedPrecisionGMRESIR

In [47]:

# --------------------------
# Example usage (uncomment to run)
# --------------------------
using MatrixMarket, SparseArrays, LinearAlgebra
using .MixedPrecisionGMRESIR

A = MatrixMarket.mmread(proj_dir * "/Matrices/af23560.mtx") |> SparseMatrixCSC{Float64,Int}
n = size(A,1)
b = ones(Float64, n)
x = MixedPrecisionGMRESIR.solve_with_ir(A, b; S=Float32, restart=50, rtol=1e-5, atol=1e-7, max_outer=20, verbose=true, jacobi=true)
@show norm(b - A*x)/norm(b)

[ Info: [IR] iter=0  ‖r‖=153.49267083479916  rel=1.0
[ Info:   [GMRES(S)] iter=1  relres=0.9992279
[ Info:   [GMRES(S)] iter=2  relres=0.9989437
[ Info:   [GMRES(S)] iter=3  relres=0.9986869
[ Info:   [GMRES(S)] iter=4  relres=0.9985399
[ Info:   [GMRES(S)] iter=5  relres=0.9983506
[ Info:   [GMRES(S)] iter=6  relres=0.9982627
[ Info:   [GMRES(S)] iter=7  relres=0.9981393
[ Info:   [GMRES(S)] iter=8  relres=0.9980767
[ Info:   [GMRES(S)] iter=9  relres=0.9979545
[ Info:   [GMRES(S)] iter=10  relres=0.9979026
[ Info:   [GMRES(S)] iter=11  relres=0.9978019
[ Info:   [GMRES(S)] iter=12  relres=0.9977536
[ Info:   [GMRES(S)] iter=13  relres=0.9976726
[ Info:   [GMRES(S)] iter=14  relres=0.9976337
[ Info:   [GMRES(S)] iter=15  relres=0.9975606
[ Info:   [GMRES(S)] iter=16  relres=0.997528
[ Info:   [GMRES(S)] iter=17  relres=0.9974596
[ Info:   [GMRES(S)] iter=18  relres=0.997428
[ Info:   [GMRES(S)] iter=19  relres=0.9973704
[ Info:   [GMRES(S)] iter=20  relres=0.9973422
[ Info:   [GMRES(S

LoadError: InterruptException:

# GMRES + IR Take 2

## Iteration 1

In [49]:
"""
    gmres_ir(A, b; pf=Float16, pw=Float32, pr=Float64,
             restart=50, max_outer=10, inner_maxiter=0,
             rtol=√(eps(pr)), atol=0.0, verbose=false)

Mixed-precision GMRES-based Iterative Refinement (GMRES-IR).

Solves A*x ≈ b with:
  - pf : factorization precision for LU preconditioner (low; e.g. Float16/Float32)
  - pw : working precision for GMRES (e.g. Float32/Float64)
  - pr : residual/accumulation precision (high; e.g. Float64/BigFloat)

`restart` is the GMRES restart length. `inner_maxiter=0` defaults to `restart`.
`rtol`/`atol` are outer stopping tolerances on ‖r‖ / ‖b‖.

Returns `(x, outer_hist)` where `outer_hist` stores the relative residuals per IR step.

References:
  - Higham (blog): “What Is Iterative Refinement?” (2023). 
  - Carson & Higham, *SIAM J. Sci. Comput.* 39(5): A2834–A2856 (2017) — GMRES-IR analysis.
  - Carson & Higham, *SIAM J. Sci. Comput.* 40(2): A817–A847 (2018) — three-precision IR.
"""
function gmres_ir(A::AbstractMatrix, b::AbstractVector;
                  pf=Float16, pw=Float32, pr=Float64,
                  restart::Int=50, max_outer::Int=10, inner_maxiter::Int=0,
                  rtol=√(eps(pr)), atol=0.0, verbose::Bool=false)

    using LinearAlgebra
    using IterativeSolvers

    n = size(A,1)
    @assert size(A,2) == n "A must be square"
    @assert length(b) == n  "b length must match A"

    # Choose inner max iters (per GMRES call)
    inner_maxiter = inner_maxiter == 0 ? restart : inner_maxiter

    # Prepare typed copies
    A_pf = Array{pf}(undef, n, n); copyto!(A_pf, pf.(A))
    A_pw = Array{pw}(undef, n, n); copyto!(A_pw, pw.(A))

    b_pr = Array{pr}(undef, n);      copyto!(b_pr, pr.(b))
    x_pr = zeros(pr, n)              # start from zero; could also warm-start via low-prec solve

    # Low-precision LU preconditioner (right preconditioning)
    lu_pf = lu(A_pf)  # pivoted LU in low precision

    # Right-preconditioner wrapper that preserves input eltype on `\`
    struct LowPrecRightPreconditioner{Tlu}
        lu::Tlu
    end
    # vector
    function Base.:\(P::LowPrecRightPreconditioner, x::AbstractVector{T}) where {T}
        xp = similar(x, eltype(P.lu.factors)); copyto!(xp, eltype(P.lu.factors).(x))
        yp = P.lu \ xp
        y  = similar(x); copyto!(y, T.(yp))
        return y
    end
    # matrix (IterativeSolvers may call this in block form)
    function Base.:\(P::LowPrecRightPreconditioner, X::AbstractMatrix{T}) where {T}
        Xp = similar(X, eltype(P.lu.factors)); copyto!(Xp, eltype(P.lu.factors).(X))
        Yp = P.lu \ Xp
        Y  = similar(X); copyto!(Y, T.(Yp))
        return Y
    end
    Pr = LowPrecRightPreconditioner(lu_pf)

    # Buffers to avoid allocations in outer loop
    Ax_pr = similar(b_pr)  # for A*x in pr
    r_pr  = similar(b_pr)  # residual in pr

    # Utility: A*x in high precision, cheaply via promotion
    function mul_pr!(y::AbstractVector{pr}, A::AbstractMatrix, x::AbstractVector{pr})
        # do the matvec in pr without converting the whole matrix each time
        # (A is any eltype; promotion happens elementwise)
        @inbounds @simd for i=1:size(A,1)
            acc = zero(pr)
            Ai = @view A[i, :]
            for j=1:size(A,2)
                acc += pr(Ai[j]) * x[j]
            end
            y[i] = acc
        end
        return y
    end

    # Outer IR loop
    β = norm(b_pr)
    β == 0 && return (zeros(pr, n), pr[0.0])  # trivial solution
    outer_hist = pr[]  # relative residual history

    for k = 1:max_outer
        mul_pr!(Ax_pr, A, x_pr)
        @. r_pr = b_pr - Ax_pr
        rel = norm(r_pr) / β
        push!(outer_hist, rel)
        verbose && @info "[IR] iter=$k  relres=$(rel)"

        if rel ≤ max(rtol, atol/β)
            break
        end

        # Solve approximately: A * (Pr \ y) ≈ r  via GMRES on A*Pr
        # Implement as GMRES on augmented operator x ↦ A_pw * (Pr \ x) in working precision.
        # Build a closure for matvec without extra deps:
        function Aeff_mv!(y::AbstractVector{pw}, x::AbstractVector{pw})
            tmp = Pr \ x        # approx A^{-1} * x, still pw (by our \ method)
            mul!(y, A_pw, tmp)  # y = A_pw * tmp
            return y
        end

        r_pw = pw.(r_pr)
        y0   = zeros(pw, n)
        y    = copy(y0)
        # workspace for matvec
        Aeff = IterativeSolvers.LinearOperator{pw}(n, n,
                    (y,x) -> Aeff_mv!(y,x),
                    (y,x) -> Aeff_mv!(y,x),  # treat adjoint same; GMRES doesn't need adjoint
                    issymmetric=false)

        # Inner GMRES tolerance — loose is usually fine for IR
        inner_rtol = pw(0.1)  # Carson–Higham recommend modest accuracy
        y, _ = gmres!(y, Aeff, r_pw; restart=restart, maxiter=inner_maxiter,
                      reltol=inner_rtol, verbose=false)

        # Map back to correction d = Pr \ y (right preconditioning)
        d_pw = Pr \ y
        # Accumulate in high precision
        @inbounds @simd for i in eachindex(x_pr)
            x_pr[i] += pr(d_pw[i])
        end
    end

    return x_pr, outer_hist
end


LoadError: syntax: local variable pr cannot be used in closure declaration

## Iteration 2

In [11]:
using LinearAlgebra
using IterativeSolvers

"""
    gmres_ir(A, b; pf=Float16, pw=Float32, pr=Float64,
             restart=50, max_outer=10, inner_maxiter=0,
             rtol=√(eps(pr)), atol=0.0, verbose=false,
             warmstart=false, form_A_high=true)

Mixed-precision GMRES-based Iterative Refinement.

- Factorization precision `pf` for low-precision LU preconditioner.
- Working precision `pw` for the inner GMRES Krylov iterations.
- Residual/accumulation precision `pr` for computing residuals and updating `x`.

Returns `(x, outer_hist)` where `outer_hist[k]` is the relative residual after IR step `k`.
"""
function gmres_ir(A::AbstractMatrix, b::AbstractVector;
                  pf=Float16, pw=Float32, pr=Float64,
                  restart::Int=50, max_outer::Int=10, inner_maxiter::Int=0,
                  rtol=√(eps(pf)), atol::Real=0.0,
                  verbose::Bool=false, warmstart::Bool=false,
                  form_A_high::Bool=true)

    n, m = size(A)
    @assert n == m "A must be square"
    @assert length(b) == n "length(b) must match size(A,1)"

    inner_maxiter = inner_maxiter == 0 ? restart : inner_maxiter

    # Typed copies (once)
    A_pf = Matrix{pf}(A)
    A_pw = Matrix{pw}(A)
    A_pr = form_A_high ? Matrix{pr}(A) : nothing  # optional to save memory
    b_pr = Vector{pr}(b)

    # Low-precision LU preconditioner
    lu_pf = lu(A_pf)

    # Initial guess (optionally warm-start with low-prec solve)
    x_pr = warmstart ? Vector{pr}(lu_pf \ Vector{pf}(b)) : zeros(pr, n)

    # Buffers
    r_pr  = similar(b_pr)
    Ax_pr = similar(b_pr)
    β = norm(b_pr)
    outer_hist = pr[]

    if β == 0
        push!(outer_hist, zero(pr))
        return x_pr, outer_hist
    end

    # Helper: residual matvec in pr (fast path uses A_pr if available)
    mul_pr! = form_A_high ?
        (y, Aany, x) -> (mul!(y, A_pr, x); y) :
        function (y, Aany, x)
            @inbounds for i in 1:n
                acc = zero(pr)
                for j in 1:n
                    acc += pr(Aany[i,j]) * x[j]
                end
                y[i] = acc
            end
            y
        end

    # Right-preconditioner application: out_pw := (LU_pf) \ x_pw, staying allocation-light
    tmp_pf = zeros(pf, n)   # reused rhs/sol in pf
    tmp_pw = zeros(pw, n)   # reused image in pw
    apply_prec! = function (out_pw, x_pw)
        @inbounds @simd for i in 1:n
            tmp_pf[i] = pf(x_pw[i])
        end
        ldiv!(lu_pf, tmp_pf)              # tmp_pf := lu_pf \ tmp_pf
        @inbounds @simd for i in 1:n
            out_pw[i] = pw(tmp_pf[i])
        end
        out_pw
    end

    # Effective operator y := A_pw * (LU_pf \ x) for GMRES
    Aeff_mv! = function (y, x)
        apply_prec!(tmp_pw, x)
        mul!(y, A_pw, tmp_pw)
        y
    end
    # Construct a LinearOperator without type-annotating local precisions in signatures
    Aeff = IterativeSolvers.LinearOperator(pw, n, n, Aeff_mv!, Aeff_mv!; issymmetric=false)

    # Inner-solve settings: loose tolerance is fine for IR
    inner_tol = pw(0.1)

    for k in 1:max_outer
        mul_pr!(Ax_pr, A, x_pr)
        @. r_pr = b_pr - Ax_pr
        rel = norm(r_pr) / β
        push!(outer_hist, rel)
        verbose && @info "[IR] iter=$k  relres=$(rel)"

        if rel ≤ max(rtol, pr(atol)/β)
            break
        end

        # Solve A*(LU^{-1} y) ≈ r  via GMRES on Aeff
        r_pw = Vector{pw}(r_pr)
        y_pw = zeros(pw, n)
        gmres!(y_pw, Aeff, r_pw; restart=restart, maxiter=inner_maxiter, tol=inner_tol, log=false)

        # Map back: d = LU^{-1} y
        d_pw = similar(y_pw)
        apply_prec!(d_pw, y_pw)

        # Accumulate in high precision
        @inbounds @simd for i in 1:n
            x_pr[i] += pr(d_pw[i])
        end
    end

    return x_pr, outer_hist
end

gmres_ir

In [12]:
A = mmread(proj_dir * "/Matrices/af23560.mtx")

23560×23560 SparseArrays.SparseMatrixCSC{Float64, Int64} with 484256 stored entries:
⎡⢿⣷⣄⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⎤
⎢⠀⠙⢿⣷⣄⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⎥
⎢⠀⠀⠀⠙⢿⣷⣄⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⎥
⎢⠀⠀⠀⠀⠀⠙⢿⣷⣄⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⎥
⎢⠀⠀⠀⠀⠀⠀⠀⠙⢿⣷⣄⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⎥
⎢⠀⠀⠀⠀⠀⠀⠀⠀⠀⠙⢿⣷⣄⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⎥
⎢⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠙⢿⣷⣄⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⎥
⎢⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠙⢿⣷⣄⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⎥
⎢⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠙⢿⣷⣄⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⎥
⎢⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠙⢿⣷⣄⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⎥
⎢⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠙⢿⣷⣄⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⎥
⎢⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠙⢿⣷⣄⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⎥
⎢⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠙⢿⣷⣄⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⎥
⎢⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠙⢿⣷⣄⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⎥
⎢⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠙⢿⣷⣄⠀⠀⠀⠀⠀⠀⠀⠀⠀⎥
⎢⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠙⢿⣷⣄⠀⠀⠀⠀⠀⠀⠀⎥
⎢⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠙⢿⣷⣄⠀⠀⠀⠀⠀⎥
⎢⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠙⢿⣷⣄⠀⠀⠀⎥
⎢⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠙⢿⣷⣄⠀⎥
⎣⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠙⢿⣷⎦

In [13]:
b = rand(size(A, 1))
x1 = A \ b
x2, _ = gmres_ir(A, b,verbose = true)
println(norm(x1 .- x2))

LoadError: InterruptException:

The code never finished for this one

In [ ]:
@btime iterative_refinement_lu($A, $b);

@btime $A \ $b;

@btime lu($A) \ $b;

## Iteration 3

In [10]:
using LinearAlgebra
using IterativeSolvers

"""
    gmres_ir(A, b; pf=Float16, pw=Float32, pr=Float64,
             restart=50, max_outer=10, inner_maxiter=0,
             rtol=√(eps(pr)), atol=0.0, verbose=false,
             warmstart=true, form_A_high=true)

Mixed-precision GMRES-based Iterative Refinement with right-preconditioned GMRES.

- pf: factorization precision for LU preconditioner (low)
- pw: working precision for GMRES (Krylov)
- pr: residual/accumulation precision (high)

Stopping uses the **componentwise backward error**
η = ‖r‖∞ / (‖A‖∞·‖x‖∞ + ‖b‖∞)
with threshold τ = max(rtol, 10*eps(pr), atol / (‖A‖∞·‖x‖∞ + ‖b‖∞)).

Returns (x_pr, outer_hist), where outer_hist[k] is η after step k.
"""
function gmres_ir(A::AbstractMatrix, b::AbstractVector;
                  pf=Float16, pw=Float32, pr=Float64,
                  restart::Int=50, max_outer::Int=10, inner_maxiter::Int=0,
                  rtol=√(eps(pw)), atol::Real=0.0, verbose::Bool=false,
                  warmstart::Bool=true, form_A_high::Bool=true)

    n, m = size(A)
    @assert n == m "A must be square"
    @assert length(b) == n "length(b) must match size(A,1)"
    inner_maxiter = inner_maxiter == 0 ? restart : inner_maxiter

    # Typed copies (one-time)
    A_pf = Matrix{pf}(A)
    A_pw = Matrix{pw}(A)
    A_pr = form_A_high ? Matrix{pr}(A) : nothing
    b_pr = Vector{pr}(b)

    # Low-precision LU preconditioner
    lu_pf = lu(A_pf)

    # Warm start helps IR a lot
    x_pr = warmstart ? Vector{pr}(lu_pf \ Vector{pf}(b)) : zeros(pr, n)

    # Precompute ‖A‖∞ in high precision once
    normA∞ = form_A_high ? norm(A_pr, Inf) :
                          pr(norm(A, Inf))  # cheap and OK

    # Buffers (reused)
    r_pr   = similar(b_pr)
    Ax_pr  = similar(b_pr)
    r_pw   = zeros(pw, n)
    y_pw   = zeros(pw, n)
    d_pw   = zeros(pw, n)
    tmp_pf = zeros(pf, n)
    tmp_pw = zeros(pw, n)

    # High-precision matvec (fast path if A_pr is available)
    mul_pr! = form_A_high ?
        (y, x) -> (mul!(y, A_pr, x); y) :
        function (y, x)
            @inbounds for i in 1:n
                acc = zero(pr)
                for j in 1:n
                    acc += pr(A[i,j]) * x[j]
                end
                y[i] = acc
            end
            y
        end

    # Right-preconditioner: out_pw := LU_pf \ x_pw   (no allocations)
    apply_prec! = function (out_pw::AbstractVector{pw}, xpw::AbstractVector{pw})
        @inbounds @simd for i in 1:n
            tmp_pf[i] = pf(xpw[i])
        end
        ldiv!(lu_pf, tmp_pf)   # tmp_pf := LU^{-1} tmp_pf
        @inbounds @simd for i in 1:n
            out_pw[i] = pw(tmp_pf[i])
        end
        out_pw
    end

    # Effective operator y := A_pw * (LU_pf \ x)   (built once)
    Aeff_mv! = function (y::AbstractVector{pw}, x::AbstractVector{pw})
        apply_prec!(tmp_pw, x)
        mul!(y, A_pw, tmp_pw)
        y
    end
    Aeff = IterativeSolvers.LinearOperator(pw, n, n, Aeff_mv!, Aeff_mv!; issymmetric=false)

    # Inner GMRES target relative tolerance:
    # pick something attainable in pw, e.g. ~sqrt(eps(pw)) or 0.1, whichever is larger
    inner_reltol = max(pw(0.1), pw(sqrt(eps(pw))))

    # Outer stopping: componentwise backward error threshold τ
    # Also guard with practical floor 10*eps(pr)
    η_threshold = pr(max(rtol, 10*eps(pr)))

    outer_hist = pr[]

    # Trivial case
    if all(==(zero(pr)), b_pr)
        push!(outer_hist, zero(pr))
        return x_pr, outer_hist
    end

    for k in 1:max_outer
        # r := b - A*x  (high precision)
        mul_pr!(Ax_pr, x_pr)
        @. r_pr = b_pr - Ax_pr

        # Componentwise backward error η
        denom = normA∞ * max(norm(x_pr, Inf), pr(1)) + norm(b_pr, Inf)
        η = denom == 0 ? norm(r_pr, Inf) : norm(r_pr, Inf) / denom
        push!(outer_hist, η)
        if verbose
            @info "[IR] iter=$(k)  η(backward)= $(η)   thresh=$(η_threshold)"
        end
        # Absolute tolerance component (rarely needed)
        if η ≤ max(η_threshold, pr(atol) / max(denom, pr(1)))
            break
        end

        # Inner right-preconditioned GMRES on Aeff:  A*(LU^{-1} y) ≈ r
        @inbounds @simd for i in 1:n
            r_pw[i] = pw(r_pr[i])
            y_pw[i] = 0
        end
        gmres!(y_pw, Aeff, r_pw; restart=restart, maxiter=inner_maxiter,
               reltol=inner_reltol, log=false)

        # Map correction: d = LU^{-1} y  (still in pw), then accumulate in pr
        apply_prec!(d_pw, y_pw)
        @inbounds @simd for i in 1:n
            x_pr[i] += pr(d_pw[i])
        end
    end

    return x_pr, outer_hist
end


LoadError: syntax: local variable pw cannot be used in closure declaration

# GMRES + IR (Carson Paper + Detailed Initial Prompt)

In [6]:
A = mmread(proj_dir * "/Matrices/af23560.mtx")

23560×23560 SparseArrays.SparseMatrixCSC{Float64, Int64} with 484256 stored entries:
⎡⢿⣷⣄⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⎤
⎢⠀⠙⢿⣷⣄⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⎥
⎢⠀⠀⠀⠙⢿⣷⣄⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⎥
⎢⠀⠀⠀⠀⠀⠙⢿⣷⣄⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⎥
⎢⠀⠀⠀⠀⠀⠀⠀⠙⢿⣷⣄⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⎥
⎢⠀⠀⠀⠀⠀⠀⠀⠀⠀⠙⢿⣷⣄⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⎥
⎢⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠙⢿⣷⣄⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⎥
⎢⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠙⢿⣷⣄⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⎥
⎢⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠙⢿⣷⣄⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⎥
⎢⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠙⢿⣷⣄⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⎥
⎢⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠙⢿⣷⣄⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⎥
⎢⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠙⢿⣷⣄⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⎥
⎢⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠙⢿⣷⣄⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⎥
⎢⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠙⢿⣷⣄⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⎥
⎢⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠙⢿⣷⣄⠀⠀⠀⠀⠀⠀⠀⠀⠀⎥
⎢⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠙⢿⣷⣄⠀⠀⠀⠀⠀⠀⠀⎥
⎢⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠙⢿⣷⣄⠀⠀⠀⠀⠀⎥
⎢⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠙⢿⣷⣄⠀⠀⠀⎥
⎢⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠙⢿⣷⣄⠀⎥
⎣⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠙⢿⣷⎦

## Iteration 1

In [4]:
using LinearAlgebra

# ---- Low-precision GMRES (no preconditioning, restart) -----------------------

function _gmres_lowprec!(A_lp::AbstractMatrix{Ti},
                         r_lp::AbstractVector{Ti},
                         V::AbstractMatrix{Ti},         # n×(m+1)
                         H::AbstractMatrix{Float64},     # (m+1)×m
                         cs::AbstractVector{Float64},
                         sn::AbstractVector{Float64},
                         w_lp::AbstractVector{Ti},       # work vector (n)
                         y::AbstractVector{Float64},     # length m
                         d_lp::AbstractVector{Ti},       # output correction (n)
                         restart::Int,
                         atol::Float64) where {Ti<:AbstractFloat}

    n = size(A_lp, 1)
    m = restart
    fill!(H, 0.0)
    fill!(cs, 0.0)
    fill!(sn, 0.0)

    # β, v1
    β = norm(r_lp)                     # computed in low precision
    if β == 0
        fill!(d_lp, zero(Ti))
        return 0, 0.0
    end
    @inbounds @views V[:, 1] .= r_lp ./ Ti(β)

    # g = β*e1 in Float64 (right-hand side of least squares)
    g = zeros(Float64, m + 1)
    g[1] = Float64(β)

    k_done = 0
    resnorm = Float64(β)

    for k in 1:m
        # w = A * v_k
        mul!(w_lp, A_lp, @view V[:, k])         # low-precision matvec

        # Arnoldi: orthogonalize against previous V
        for j in 1:k
            hj = dot(@view(V[:, j]), w_lp)      # low-precision dot
            H[j, k] = Float64(hj)
            @inbounds @views w_lp .-= Ti(hj) .* V[:, j]
        end
        hk1 = norm(w_lp)                        # low-precision norm
        H[k+1, k] = Float64(hk1)

        # v_{k+1}
        if hk1 != 0
            @inbounds @views V[:, k+1] .= w_lp ./ Ti(hk1)
        else
            # happy breakdown
            @inbounds @views V[:, k+1] .= zero(Ti)
        end

        # Apply previous Givens to the new column
        for j in 1:k-1
            c = cs[j]; s = sn[j]
            h1 = H[j, k]; h2 = H[j+1, k]
            H[j, k]   =  c*h1 + s*h2
            H[j+1, k] = -s*h1 + c*h2
        end

        # New Givens for (k,k) and (k+1,k)
        h_kk  = H[k, k]
        h_k1k = H[k+1, k]
        denom = hypot(h_kk, h_k1k) + eps(Float64)
        c = h_kk / denom
        s = h_k1k / denom
        cs[k] = c
        sn[k] = s

        # Apply to H and g
        H[k, k]   =  c*h_kk + s*h_k1k
        H[k+1, k] = 0.0
        gk  = g[k]
        gk1 = g[k+1]
        g[k]   =  c*gk + s*gk1
        g[k+1] = -s*gk + c*gk1

        resnorm = abs(g[k+1])
        k_done = k

        if resnorm ≤ atol
            break
        end
    end

    # Solve upper-triangular R y = g(1:k)
    @inbounds for i in k_done:-1:1
        ssum = 0.0
        @inbounds for j in i+1:k_done
            ssum += H[i, j] * y[j]
        end
        y[i] = (g[i] - ssum) / H[i, i]
    end

    # Form d = V(:,1:k)*y in low precision
    fill!(d_lp, zero(Ti))
    @inbounds for j in 1:k_done
        α = Ti(y[j])
        @inbounds @views d_lp .+= α .* V[:, j]
    end

    return k_done, resnorm
end

# ---- Mixed-precision Iterative Refinement (outer) ----------------------------

"""
    x, stats = gmres_ir(A, b; inner=Float32, tol=1e-12, maxiters=10, restart=50, verbose=false)

Solve `Ax = b` using mixed-precision iterative refinement:
- Outer arithmetic & residuals in Float64
- Inner correction solves by **GMRES** in low precision (`inner`, Float32/Float16)
- Initial guess from a single low-precision LU solve (factorized once)

Arguments:
- `A::AbstractMatrix`, `b::AbstractVector`
- `inner`: low precision for GMRES & initial LU (default `Float32`)
- `tol`: stopping on `‖r‖₂ / (‖A‖₁‖x‖∞ + ‖b‖₂) ≤ tol` (computed in Float64)
- `maxiters`: max outer refinement steps
- `restart`: GMRES restart parameter
- `verbose`: print per-iteration info

Returns `(x::Vector{Float64}, stats::NamedTuple)`
"""
function gmres_ir(A, b; inner::Type{<:AbstractFloat}=Float32,
                  tol::Float64=1e-12, maxiters::Int=10, restart::Int=50, verbose::Bool=false)

    # Dimensions & basic checks
    n = size(A, 1)
    @assert size(A, 2) == n "A must be square"
    @assert length(b) == n  "b has incompatible length"
    @assert restart ≥ 1

    # Promote to explicit arrays (stable views/mul!)
    A64 = Array{Float64}(undef, n, n); copyto!(A64, A)
    b64 = Array{Float64}(undef, n);    copyto!(b64, b)

    # Low-precision copies for inner work
    A_lp = Array{inner}(undef, n, n);  copyto!(A_lp, A64)
    b_lp = Array{inner}(undef, n);     copyto!(b_lp, b64)

    # Initial guess: single LU factorization in low precision, used ONCE
    F = lu(A_lp)                         # factorize once
    x_lp = F \ b_lp                      # low-precision solve
    x64  = Array{Float64}(undef, n);    copyto!(x64, x_lp)   # working solution in Float64

    # Preallocate GMRES workspaces (reused across outer iterations)
    m = restart
    V   = Array{inner}(undef, n, m+1)
    H   = Array{Float64}(undef, m+1, m)
    cs  = Array{Float64}(undef, m)
    sn  = Array{Float64}(undef, m)
    wlp = Array{inner}(undef, n)
    y   = Array{Float64}(undef, m)
    dlp = Array{inner}(undef, n)

    # Norms for scaled stopping criterion
    Anorm1 = opnorm(A64, 1)
    bnorm2 = norm(b64)

    history_relres = Float64[]
    num_matvecs = 0
    total_inner_iters = 0

    for k in 0:maxiters
        # Residual in Float64
        r64 = b64 - A64 * x64
        rnorm2 = norm(r64)
        denom  = Anorm1 * maximum(abs, x64) + bnorm2
        relres = (denom == 0.0) ? 0.0 : (rnorm2 / denom)
        push!(history_relres, relres)

        verbose && @info "[IR] iter=$(k)  ‖r‖₂=$(rnorm2)  rel=$(relres)"

        if relres ≤ tol
            break
        end
        if k == maxiters
            break
        end

        # Solve A * d = r by low-precision GMRES (no preconditioner)
        copyto!(wlp, r64)                 # wlp temp used to cast
        r_lp = wlp; @inbounds @views r_lp .= inner.(r64)

        # Zero outputs & y
        fill!(dlp, zero(inner))
        fill!(y, 0.0)

        inner_iters, arnoldi_res = _gmres_lowprec!(A_lp, r_lp, V, H, cs, sn, wlp, y, dlp, m, tol*bnorm2)
        total_inner_iters += inner_iters
        num_matvecs += inner_iters

        # Update x in Float64 with low-precision correction
        @inbounds @views x64 .+= Float64.(dlp)
    end

    stats = (relres = last(history_relres),
             history = history_relres,
             inner_iters = total_inner_iters,
             matvecs = num_matvecs,
             restart = restart,
             inner = inner)

    return x64, stats
end

gmres_ir

In [10]:
b = rand(size(A, 1))
x2, stats = gmres_ir(A, b,inner=Float32, tol=1e-12, maxiters=10, restart=50, verbose=true)
x1 = A \ b
println(norm(x1 .- x2))

[ Info: [IR] iter=0  ‖r‖₂=0.020575755660624365  rel=4.256415254348543e-7
[ Info: [IR] iter=1  ‖r‖₂=0.0018646775381868446  rel=3.85736832027773e-8
[ Info: [IR] iter=2  ‖r‖₂=0.0007281166803592036  rel=1.506219439117741e-8
[ Info: [IR] iter=3  ‖r‖₂=0.0004886760998580885  rel=1.010900205050743e-8
[ Info: [IR] iter=4  ‖r‖₂=0.00038845611812474617  rel=8.035799436539968e-9
[ Info: [IR] iter=5  ‖r‖₂=0.0003217106544247598  rel=6.6550683570550625e-9
[ Info: [IR] iter=6  ‖r‖₂=0.0002833358712541432  rel=5.861227970427273e-9
[ Info: [IR] iter=7  ‖r‖₂=0.00025586875463993184  rel=5.293029482567016e-9
[ Info: [IR] iter=8  ‖r‖₂=0.00023663068403368027  rel=4.895061438197382e-9
[ Info: [IR] iter=9  ‖r‖₂=0.00022138482466643055  rel=4.579678393265707e-9
[ Info: [IR] iter=10  ‖r‖₂=0.00020271372528087476  rel=4.193439210701421e-9


0.0004058413161900606


In [9]:
@btime gmres_ir($A, $b,inner=Float32, tol=1e-12, maxiters=10, restart=50, verbose=false);

@btime $A \ $b;

  59.509 s (91 allocations: 8.28 GiB)
  180.937 ms (87 allocations: 151.94 MiB)


## Iteration 2

In [11]:
using LinearAlgebra
using SparseArrays

# ---------- Low-precision GMRES (restart, no preconditioner) ------------------

function _gmres_lowprec!(A_lp::AbstractMatrix{Ti},
                         r_lp::AbstractVector{Ti},
                         V::AbstractMatrix{Ti},         # n×(m+1)
                         H::AbstractMatrix{Float64},     # (m+1)×m
                         cs::AbstractVector{Float64},
                         sn::AbstractVector{Float64},
                         w_lp::AbstractVector{Ti},       # work vector (n)
                         y::AbstractVector{Float64},     # length m
                         d_lp::AbstractVector{Ti},       # output correction (n)
                         restart::Int,
                         atol::Float64) where {Ti<:AbstractFloat}

    n = size(A_lp, 1)
    m = restart
    fill!(H, 0.0); fill!(cs, 0.0); fill!(sn, 0.0)

    β = norm(r_lp)                     # low-precision norm
    if β == 0
        fill!(d_lp, zero(Ti))
        return 0, 0.0
    end
    @inbounds @views V[:, 1] .= r_lp ./ Ti(β)

    # g = β*e1 (in Float64)
    g = zeros(Float64, m + 1)
    g[1] = Float64(β)

    k_done = 0
    resnorm = Float64(β)

    for k in 1:m
        mul!(w_lp, A_lp, @view V[:, k])  # w = A*v_k (low-precision)

        # Arnoldi
        @inbounds for j in 1:k
            hj = dot(@view(V[:, j]), w_lp)       # low-precision dot
            H[j, k] = Float64(hj)
            @views w_lp .-= Ti(hj) .* V[:, j]
        end
        hk1 = norm(w_lp)                         # low-precision norm
        H[k+1, k] = Float64(hk1)

        if hk1 != 0
            @views V[:, k+1] .= w_lp ./ Ti(hk1)
        else
            @views V[:, k+1] .= zero(Ti)
        end

        # Apply previous Givens
        @inbounds for j in 1:k-1
            c = cs[j]; s = sn[j]
            h1 = H[j, k]; h2 = H[j+1, k]
            H[j, k]   =  c*h1 + s*h2
            H[j+1, k] = -s*h1 + c*h2
        end

        # New Givens
        h_kk  = H[k, k]; h_k1k = H[k+1, k]
        denom = hypot(h_kk, h_k1k) + eps(Float64)
        c = h_kk / denom
        s = h_k1k / denom
        cs[k] = c; sn[k] = s

        H[k, k]   =  c*h_kk + s*h_k1k
        H[k+1, k] = 0.0

        gk  = g[k]; gk1 = g[k+1]
        g[k]   =  c*gk + s*gk1
        g[k+1] = -s*gk + c*gk1

        resnorm = abs(g[k+1])
        k_done = k
        if resnorm ≤ atol
            break
        end
    end

    # Back solve R*y = g(1:k_done)
    @inbounds for i in k_done:-1:1
        ssum = 0.0
        @inbounds for j in i+1:k_done
            ssum += H[i, j] * y[j]
        end
        y[i] = (g[i] - ssum) / H[i, i]
    end

    # d = V(:,1:k_done) * y  (low precision)
    fill!(d_lp, zero(Ti))
    @inbounds for j in 1:k_done
        α = Ti(y[j])
        @views d_lp .+= α .* V[:, j]
    end

    return k_done, resnorm
end

# ---------- Mixed-precision IR (sparsity-preserving) --------------------------

"""
    x, stats = gmres_ir(A, b; inner=Float32, tol=1e-12, maxiters=10, restart=50,
                        init=:zeros, verbose=false)

Sparse/dense friendly mixed-precision GMRES-based iterative refinement.

- Outer arithmetic & residuals in Float64 (with `mul!` to avoid allocations).
- Inner solves (GMRES) in low precision `inner` (Float32/Float16).
- Preserves sparsity: the low-precision operator `A_lp` has the same sparsity pattern.
- Initial guess:
    * `init = :zeros` (default) uses x₀ = 0 (no factorization).
    * `init = :lu` does a single low-precision sparse LU solve to form x₀ (once).

Returns `(x::Vector{Float64}, stats::NamedTuple)`.
"""
function gmres_ir(A, b; inner::Type{<:AbstractFloat}=Float32,
                  tol::Float64=1e-12, maxiters::Int=10, restart::Int=50,
                  init::Symbol=:zeros, verbose::Bool=false)

    n = size(A, 1)
    @assert size(A, 2) == n "A must be square"
    @assert length(b) == n  "b has incompatible length"
    @assert restart ≥ 1

    # b and working solution in Float64
    b64 = Array{Float64}(undef, n); copyto!(b64, b)
    x64 = zeros(Float64, n)

    # Low-precision operator with same sparsity pattern (or dense if A is dense)
    A_lp = issparse(A) ? convert(SparseMatrixCSC{inner, Int}, A) :
                         Array{inner}(A)  # dense path only if input is dense

    # Optional single low-precision LU for initial guess
    if init === :lu
        if issparse(A_lp)
            F = lu(A_lp)             # SuiteSparse LU in low precision (once)
            x0_lp = F \ convert(Vector{inner}, b64)
        else
            F = lu(A_lp)
            x0_lp = F \ convert(Vector{inner}, b64)
        end
        @inbounds @views x64 .= Float64.(x0_lp)
    elseif init === :zeros
        # x64 already zeros
        nothing
    else
        error("init must be :zeros or :lu")
    end

    # Preallocate outer work
    r64  = similar(b64)   # residual
    Ax64 = similar(b64)   # A*x in Float64

    # Preallocate GMRES work
    m   = restart
    V   = Array{inner}(undef, n, m+1)
    H   = Array{Float64}(undef, m+1, m)
    cs  = Array{Float64}(undef, m)
    sn  = Array{Float64}(undef, m)
    wlp = Array{inner}(undef, n)
    y   = Array{Float64}(undef, m)
    dlp = Array{inner}(undef, n)
    rlp = Array{inner}(undef, n)

    # Norms for scaled stopping criterion
    # Note: opnorm(A,1) for sparse uses SuiteSparse and is efficient
    Anorm1 = opnorm(A, 1) |> Float64
    bnorm2 = norm(b64)

    history_relres = Float64[]
    matvecs = 0
    total_inner_iters = 0

    for k in 0:maxiters
        # r64 = b64 - A*x64, all in Float64 with mul!
        mul!(Ax64, A, x64)       # Ax in Float64
        @inbounds @views r64 .= b64 .- Ax64

        rnorm2 = norm(r64)
        denom  = Anorm1 * maximum(abs, x64) + bnorm2
        relres = (denom == 0.0) ? 0.0 : (rnorm2 / denom)
        push!(history_relres, relres)

        verbose && @info "[IR] iter=$(k)  ‖r‖₂=$(rnorm2)  rel=$(relres)"

        if relres ≤ tol
            break
        end
        if k == maxiters
            break
        end

        # Cast residual to low precision (no new allocs)
        @inbounds @views rlp .= inner.(r64)

        # Zero outputs
        fill!(dlp, zero(inner)); fill!(y, 0.0)

        # Inner GMRES tolerance (absolute on residual). Keep simple & safe.
        atol_inner = max(tol * bnorm2, 1e-2 * rnorm2)

        inner_iters, _ = _gmres_lowprec!(A_lp, rlp, V, H, cs, sn, wlp, y, dlp, m, atol_inner)
        total_inner_iters += inner_iters
        matvecs += inner_iters

        # x64 += Float64.(dlp)  (no new allocs)
        @inbounds @views x64 .+= Float64.(dlp)
    end

    stats = (relres = last(history_relres),
             history = history_relres,
             inner_iters = total_inner_iters,
             matvecs = matvecs,
             restart = restart,
             inner = inner,
             init = init)

    return x64, stats
end


gmres_ir

In [12]:
b = randn(size(A, 1))
x2, stats = gmres_ir(A, b,inner=Float32, tol=1e-12, maxiters=10, restart=50, verbose=true)
x1 = A \ b
println(norm(x1 .- x2))

[ Info: [IR] iter=0  ‖r‖₂=153.8377502491046  rel=1.0
[ Info: [IR] iter=1  ‖r‖₂=30.573963352579227  rel=0.06273511515604811
[ Info: [IR] iter=2  ‖r‖₂=20.35675648439881  rel=0.03108987311557452
[ Info: [IR] iter=3  ‖r‖₂=15.712178192220584  rel=0.0182668287846715
[ Info: [IR] iter=4  ‖r‖₂=13.56195879643487  rel=0.013128373261870687
[ Info: [IR] iter=5  ‖r‖₂=12.84158342962825  rel=0.011224118853883596
[ Info: [IR] iter=6  ‖r‖₂=12.519890575877097  rel=0.010253310006176818
[ Info: [IR] iter=7  ‖r‖₂=12.281778282310716  rel=0.009571772061962924
[ Info: [IR] iter=8  ‖r‖₂=12.025042794715455  rel=0.008954431452494254
[ Info: [IR] iter=9  ‖r‖₂=11.764843396285428  rel=0.008405270091667999
[ Info: [IR] iter=10  ‖r‖₂=11.517974685492337  rel=0.007960772688806694


39.287023366275


In [13]:
b = rand(size(A, 1))
x2, stats = gmres_ir(A, b,inner=Float32, tol=1e-12, maxiters=10, restart=50, verbose=true, init =:lu)
x1 = A \ b
println(norm(x1 .- x2))

[ Info: [IR] iter=0  ‖r‖₂=0.0009293096291680524  rel=1.911253762116067e-8
[ Info: [IR] iter=1  ‖r‖₂=0.0001460172256976641  rel=3.00304607385941e-9
[ Info: [IR] iter=2  ‖r‖₂=9.726605554036064e-5  rel=2.000410886003964e-9
[ Info: [IR] iter=3  ‖r‖₂=7.581161706669703e-5  rel=1.5591707230520878e-9
[ Info: [IR] iter=4  ‖r‖₂=6.415593491589035e-5  rel=1.3194555148886314e-9
[ Info: [IR] iter=5  ‖r‖₂=5.5623059037114104e-5  rel=1.143965125018695e-9
[ Info: [IR] iter=6  ‖r‖₂=5.0465907248083855e-5  rel=1.037901159924488e-9
[ Info: [IR] iter=7  ‖r‖₂=4.539863541799748e-5  rel=9.33685699144587e-10
[ Info: [IR] iter=8  ‖r‖₂=4.123189836720788e-5  rel=8.47990983241178e-10
[ Info: [IR] iter=9  ‖r‖₂=3.6948193645887905e-5  rel=7.598906672815019e-10
[ Info: [IR] iter=10  ‖r‖₂=2.8795142193582383e-5  rel=5.922118909998257e-10


3.933641633899986e-5


In [22]:
@btime gmres_ir($A, $b,inner=Float32, tol=1e-12, maxiters=10, restart=50, verbose=false);
@btime gmres_ir($A, $b,inner=Float32, tol=1e-12, maxiters=10, restart=50, verbose=false, init = :lu);
@btime $A \ $b;
@btime lu($A) \ $b;

  212.112 ms (547 allocations: 11.34 MiB)
  404.396 ms (640 allocations: 170.76 MiB)
  178.150 ms (87 allocations: 151.94 MiB)
  180.534 ms (85 allocations: 151.76 MiB)


## Iteration 3

In [23]:
using LinearAlgebra
using SparseArrays

# ---------------------- Helpers: Float64 inner products -----------------------

@inline function dot64(x::AbstractVector{<:AbstractFloat}, y::AbstractVector{<:AbstractFloat})
    @assert length(x) == length(y)
    s = 0.0
    @inbounds @simd for i in eachindex(x, y)
        s += Float64(x[i]) * Float64(y[i])
    end
    return s
end

@inline function nrm2_64(x::AbstractVector{<:AbstractFloat})
    return sqrt(dot64(x, x))
end

# ----------------- Low-precision GMRES with MGS-64 (no precond) ---------------

function _gmres_lowprec!(A_lp::AbstractMatrix{Ti},
                         r_lp::AbstractVector{Ti},
                         V::AbstractMatrix{Ti},         # n×(m+1)
                         H::AbstractMatrix{Float64},     # (m+1)×m
                         cs::AbstractVector{Float64},
                         sn::AbstractVector{Float64},
                         g::AbstractVector{Float64},     # length m+1
                         w_lp::AbstractVector{Ti},       # work vector (n)
                         y::AbstractVector{Float64},     # length m
                         d_lp::AbstractVector{Ti},       # output correction (n)
                         restart::Int,
                         atol::Float64) where {Ti<:AbstractFloat}

    n = size(A_lp, 1)
    m = restart
    fill!(H, 0.0); fill!(cs, 0.0); fill!(sn, 0.0); fill!(g, 0.0)

    β = nrm2_64(r_lp)                  # compute norm in Float64
    if β == 0.0
        fill!(d_lp, zero(Ti))
        return 0, 0.0
    end
    @inbounds @views V[:, 1] .= r_lp ./ Ti(β)
    g[1] = β

    k_done = 0
    resnorm = β

    for k in 1:m
        mul!(w_lp, A_lp, @view V[:, k])  # w = A*v_k  (low-precision storage)

        # Modified Gram–Schmidt with Float64 dots
        @inbounds for j in 1:k
            hj = dot64(@view(V[:, j]), w_lp)
            H[j, k] = hj
            @views w_lp .-= Ti(hj) .* V[:, j]
        end
        hk1 = nrm2_64(w_lp)
        H[k+1, k] = hk1

        if hk1 != 0.0
            @views V[:, k+1] .= w_lp ./ Ti(hk1)
        else
            @views V[:, k+1] .= zero(Ti) # happy breakdown
        end

        # Apply previous Givens to H(:,k)
        @inbounds for j in 1:k-1
            c = cs[j]; s = sn[j]
            h1 = H[j, k]; h2 = H[j+1, k]
            H[j, k]   =  c*h1 + s*h2
            H[j+1, k] = -s*h1 + c*h2
        end

        # New Givens for (k,k) and (k+1,k)
        h_kk  = H[k, k]; h_k1k = H[k+1, k]
        denom = hypot(h_kk, h_k1k) + eps(Float64)
        c = h_kk / denom
        s = h_k1k / denom
        cs[k] = c; sn[k] = s

        H[k, k]   =  c*h_kk + s*h_k1k
        H[k+1, k] = 0.0

        # Update g
        gk  = g[k]; gk1 = g[k+1]
        g[k]   =  c*gk + s*gk1
        g[k+1] = -s*gk + c*gk1

        resnorm = abs(g[k+1])
        k_done = k
        if resnorm ≤ atol
            break
        end
    end

    # Backsolve R*y = g(1:k_done)
    @inbounds for i in k_done:-1:1
        ssum = 0.0
        @inbounds for j in i+1:k_done
            ssum += H[i, j] * y[j]
        end
        y[i] = (g[i] - ssum) / H[i, i]
    end

    # d = V(:,1:k_done) * y  (accumulate in low precision)
    fill!(d_lp, zero(Ti))
    @inbounds for j in 1:k_done
        α = Ti(y[j])
        @views d_lp .+= α .* V[:, j]
    end

    return k_done, resnorm
end

# ---------------- Mixed-precision IR (sparse-friendly, tuned) -----------------

"""
    x, stats = gmres_ir(A, b; inner=Float32, tol=1e-12, maxiters=10,
                        restart=30, init=:diag, verbose=false)

- Outer arithmetic & residuals in Float64 (`mul!`, no allocs).
- Inner GMRES in `inner` precision (Float32/Float16) with **Float64 orthogonalization**.
- Initial guess:
    * `:diag` (default): x0_i = b_i / A_ii if A_ii≠0, else 0  (cheap & robust)
    * `:lu`  : single low-precision sparse LU solve to form x0  (more cost)
    * `:zeros`: x0 = 0  (may stall on difficult matrices)
- No preconditioning (unless you ask later).

Returns `(x::Vector{Float64}, stats::NamedTuple)`.
"""
function gmres_ir(A, b; inner::Type{<:AbstractFloat}=Float32,
                  tol::Float64=1e-12, maxiters::Int=10, restart::Int=30,
                  init::Symbol=:diag, verbose::Bool=false)

    n = size(A, 1)
    @assert size(A, 2) == n "A must be square"
    @assert length(b) == n  "b has incompatible length"
    @assert restart ≥ 1

    # Float64 work
    b64 = Array{Float64}(undef, n); copyto!(b64, b)
    x64 = zeros(Float64, n)
    r64 = similar(b64)
    Ax64 = similar(b64)

    # Low-precision operator (preserve sparsity)
    A_lp = issparse(A) ? convert(SparseMatrixCSC{inner, Int}, A) :
                         Array{inner}(A)

    # Initial guess
    if init === :diag
        # x0_i = b_i / A_ii when possible (cheap diagonal solve)
        if issparse(A)
            D = diag(A)
            @inbounds for i in eachindex(x64)
                di = D[i]
                x64[i] = (di != 0) ? (b64[i] / Float64(di)) : 0.0
            end
        else
            @inbounds for i in 1:n
                di = A[i, i]
                x64[i] = (di != 0) ? (b64[i] / Float64(di)) : 0.0
            end
        end
    elseif init === :lu
        F = lu(A_lp)
        b_lp = similar(b64, inner); @inbounds b_lp .= inner.(b64)
        x0_lp = F \ b_lp
        @inbounds @views x64 .= Float64.(x0_lp)
    elseif init === :zeros
        # already zeros
        nothing
    else
        error("init must be :diag, :lu, or :zeros")
    end

    # GMRES workspaces
    m   = restart
    V   = Array{inner}(undef, n, m+1)
    H   = Array{Float64}(undef, m+1, m)
    cs  = Array{Float64}(undef, m)
    sn  = Array{Float64}(undef, m)
    g   = Array{Float64}(undef, m+1)
    wlp = Array{inner}(undef, n)
    y   = Array{Float64}(undef, m)
    dlp = Array{inner}(undef, n)
    rlp = Array{inner}(undef, n)

    # Stopping scale
    Anorm1 = opnorm(A, 1) |> Float64
    bnorm2 = norm(b64)

    history_relres = Float64[]
    matvecs = 0
    total_inner_iters = 0

    for k in 0:maxiters
        # r64 = b - A*x (Float64, allocation-free)
        mul!(Ax64, A, x64)
        @inbounds @views r64 .= b64 .- Ax64

        rnorm2 = norm(r64)
        denom  = Anorm1 * maximum(abs, x64) + bnorm2
        relres = (denom == 0.0) ? 0.0 : (rnorm2 / denom)
        push!(history_relres, relres)
        verbose && @info "[IR] iter=$(k)  ‖r‖₂=$(rnorm2)  rel=$(relres)"

        if relres ≤ tol || k == maxiters
            break
        end

        # Cast residual to low precision (no allocations)
        @inbounds @views rlp .= inner.(r64)

        # Inner GMRES: stronger target to reduce number of outer steps
        atol_inner = max(1e-3*rnorm2, tol*bnorm2)

        # Zero outputs
        fill!(dlp, zero(inner)); fill!(y, 0.0); fill!(g, 0.0)

        inner_iters, _ =
            _gmres_lowprec!(A_lp, rlp, V, H, cs, sn, g, wlp, y, dlp, m, atol_inner)

        total_inner_iters += inner_iters
        matvecs += inner_iters

        # Update x
        @inbounds @views x64 .+= Float64.(dlp)
    end

    stats = (relres = last(history_relres),
             history = history_relres,
             inner_iters = total_inner_iters,
             matvecs = matvecs,
             restart = restart,
             inner = inner,
             init = init)

    return x64, stats
end


gmres_ir

In [27]:
b = randn(size(A, 1))
x2, stats = gmres_ir(A, b,inner=Float32, tol=1e-12, maxiters=10, restart=50, verbose=true, init = :diag)
x1 = A \ b
println(norm(x1 .- x2))

[ Info: [IR] iter=0  ‖r‖₂=199.40213750671435  rel=0.6228390397835848
[ Info: [IR] iter=1  ‖r‖₂=33.219209399460716  rel=0.0625951132192568
[ Info: [IR] iter=2  ‖r‖₂=23.568161998091778  rel=0.03400707666702436
[ Info: [IR] iter=3  ‖r‖₂=19.30412197374531  rel=0.02261818981118923
[ Info: [IR] iter=4  ‖r‖₂=17.06060620530942  rel=0.0179052544347327
[ Info: [IR] iter=5  ‖r‖₂=15.7559702315151  rel=0.015425849753413007
[ Info: [IR] iter=6  ‖r‖₂=14.473516548192013  rel=0.013279280260170738
[ Info: [IR] iter=7  ‖r‖₂=13.476505333927447  rel=0.011932552217410852
[ Info: [IR] iter=8  ‖r‖₂=13.120908929964918  rel=0.011508132456638053
[ Info: [IR] iter=9  ‖r‖₂=13.019515834162382  rel=0.011367791668615252
[ Info: [IR] iter=10  ‖r‖₂=12.96782021387378  rel=0.011318231713724862


64.68623971517317


In [25]:
b = rand(size(A, 1))
x2, stats = gmres_ir(A, b,inner=Float32, tol=1e-12, maxiters=10, restart=50, verbose=true, init =:lu)
x1 = A \ b
println(norm(x1 .- x2))

[ Info: [IR] iter=0  ‖r‖₂=0.0009161630535203236  rel=1.9309279690453815e-8
[ Info: [IR] iter=1  ‖r‖₂=0.00014397800694052718  rel=3.0345160818962846e-9
[ Info: [IR] iter=2  ‖r‖₂=9.584920571968378e-5  rel=2.0201415765063556e-9
[ Info: [IR] iter=3  ‖r‖₂=7.47904175557063e-5  rel=1.576301365644055e-9
[ Info: [IR] iter=4  ‖r‖₂=6.340178420445518e-5  rel=1.3362717169266332e-9
[ Info: [IR] iter=5  ‖r‖₂=5.521609458306897e-5  rel=1.163748094091276e-9
[ Info: [IR] iter=6  ‖r‖₂=5.032926579349177e-5  rel=1.0607520719973432e-9
[ Info: [IR] iter=7  ‖r‖₂=4.530910489972971e-5  rel=9.549459075660106e-10
[ Info: [IR] iter=8  ‖r‖₂=4.124449471506982e-5  rel=8.69279168762575e-10
[ Info: [IR] iter=9  ‖r‖₂=3.719654567296799e-5  rel=7.839635860157385e-10
[ Info: [IR] iter=10  ‖r‖₂=2.9313177071569198e-5  rel=6.178117551395592e-10


3.854637073952677e-5


In [26]:
@btime gmres_ir($A, $b,inner=Float32, tol=1e-12, maxiters=10, restart=50, verbose=false);
@btime gmres_ir($A, $b,inner=Float32, tol=1e-12, maxiters=10, restart=50, verbose=false, init = :lu);
@btime $A \ $b;
@btime lu($A) \ $b;

  216.513 ms (556 allocations: 11.98 MiB)
  406.103 ms (631 allocations: 170.76 MiB)
  180.124 ms (87 allocations: 151.94 MiB)
  181.890 ms (85 allocations: 151.76 MiB)


## Iteration 3

In [34]:
using LinearAlgebra
using SparseArrays

#----------------- utility: Float64 dot/norm (no @simd for Apple) --------------

@inline function dot64(x::AbstractVector{<:AbstractFloat}, y::AbstractVector{<:AbstractFloat})
    s = 0.0
    @inbounds for i in eachindex(x, y)
        s += Float64(x[i]) * Float64(y[i])
    end
    return s
end

@inline nrm2_64(x::AbstractVector{<:AbstractFloat}) = sqrt(dot64(x, x))

#----------------- low-precision GMRES (restart) with MGS-64 -------------------

function _gmres_lowprec!(A_lp::AbstractMatrix{Ti},
                         r_lp::AbstractVector{Ti},
                         V::AbstractMatrix{Ti},         # n×(m+1)
                         H::AbstractMatrix{Float64},     # (m+1)×m
                         cs::AbstractVector{Float64},
                         sn::AbstractVector{Float64},
                         g::AbstractVector{Float64},     # length m+1
                         w_lp::AbstractVector{Ti},       # work (n)
                         y::AbstractVector{Float64},     # length m
                         d_lp::AbstractVector{Ti},       # output correction (n)
                         restart::Int,
                         atol::Float64) where {Ti<:AbstractFloat}

    m = restart
    fill!(H, 0.0); fill!(cs, 0.0); fill!(sn, 0.0); fill!(g, 0.0)

    β = nrm2_64(r_lp)
    if β == 0.0
        fill!(d_lp, zero(Ti))
        return 0, 0.0
    end
    @inbounds @views V[:,1] .= r_lp ./ Ti(β)
    g[1] = β

    k_done = 0
    resn = β

    for k in 1:m
        mul!(w_lp, A_lp, @view V[:,k])                 # w = A*v_k (low-prec)

        @inbounds for j in 1:k                         # MGS with 64-bit dots
            hj = dot64(@view(V[:,j]), w_lp)
            H[j,k] = hj
            @views w_lp .-= Ti(hj) .* V[:,j]
        end
        hk1 = nrm2_64(w_lp)
        H[k+1,k] = hk1

        if hk1 != 0.0
            @views V[:,k+1] .= w_lp ./ Ti(hk1)
        else
            @views V[:,k+1] .= zero(Ti)                # happy breakdown
        end

        # apply previous Givens
        @inbounds for j in 1:k-1
            c = cs[j]; s = sn[j]
            h1 = H[j,k]; h2 = H[j+1,k]
            H[j,k]   =  c*h1 + s*h2
            H[j+1,k] = -s*h1 + c*h2
        end

        # new Givens
        hkk  = H[k,k]; hk1k = H[k+1,k]
        den = hypot(hkk, hk1k) + eps(Float64)
        c = hkk/den; s = hk1k/den
        cs[k] = c; sn[k] = s
        H[k,k]   =  c*hkk + s*hk1k
        H[k+1,k] =  0.0

        # update g
        gk  = g[k]; gk1 = g[k+1]
        g[k]   =  c*gk + s*gk1
        g[k+1] = -s*gk + c*gk1

        resn = abs(g[k+1])
        k_done = k
        if resn ≤ atol
            break
        end
    end

    # backsolve R*y = g(1:k_done)
    @inbounds for i in k_done:-1:1
        ssum = 0.0
        @inbounds for j in i+1:k_done
            ssum += H[i,j] * y[j]
        end
        y[i] = (g[i] - ssum) / H[i,i]
    end

    # d = V(:,1:k_done) * y  (accumulate in low precision, no temps)
    fill!(d_lp, zero(Ti))
    @inbounds for j in 1:k_done
        α = Ti(y[j])
        @views d_lp .+= α .* V[:,j]
    end

    return k_done, resn
end

#----------------- mixed-precision IR (sparse-friendly, lean) ------------------

"""
    x, stats = gmres_ir(A, b; inner=Float32, tol=1e-12, maxiters=6,
                        restart=30, init=:lu, scale=:bnorm, verbose=false)

- Outer arithmetic & residuals in Float64 (`mul!`, no allocs).
- Inner GMRES in `inner` precision (Float32/Float16) with Float64 MGS.
- `init`:
    * `:lu` (default): one low-precision sparse LU **only to form x₀** (never reused).
    * `:zeros`, `:diag` available but generally not robust across your cases.
- `scale`: stopping scale for the IR outer loop.
    * `:bnorm` (default): uses `‖r‖₂ / (‖b‖₂ + ϵ)` — cheap and effective for timing.
    * `:opnorm`: uses `‖r‖₂ / (‖A‖₁‖x‖∞ + ‖b‖₂)` — stricter & costlier once at setup.

Returns `(x::Vector{Float64}, stats::NamedTuple)`.
"""
function gmres_ir(A, b; inner::Type{<:AbstractFloat}=Float32,
                  tol::Float64=1e-12, maxiters::Int=6, restart::Int=30,
                  init::Symbol=:lu, scale::Symbol=:bnorm, verbose::Bool=false)

    n = size(A,1)
    @assert size(A,2) == n  "A must be square"
    @assert length(b) == n  "b has incompatible length"
    @assert restart ≥ 1

    # Float64 work
    b64 = Array{Float64}(undef, n); copyto!(b64, b)
    x64 = zeros(Float64, n)
    r64 = similar(b64)
    Ax64 = similar(b64)

    # Low-precision operator (preserve sparsity)
    A_lp = issparse(A) ? convert(SparseMatrixCSC{inner,Int}, A) : Array{inner}(A)

    # Initial guess (robust default)
    if init === :lu
        F = lu(A_lp)                                   # factorize ONCE
        b_lp = Vector{inner}(undef, n); @inbounds b_lp .= inner.(b64)
        x0_lp = F \ b_lp
        @inbounds for i in 1:n
            x64[i] = Float64(x0_lp[i])                # no broadcast temp
        end
    elseif init === :diag
        if issparse(A)
            D = diag(A)
            @inbounds for i in 1:n
                di = D[i]
                x64[i] = (di != 0) ? b64[i]/Float64(di) : 0.0
            end
        else
            @inbounds for i in 1:n
                di = A[i,i]
                x64[i] = (di != 0) ? b64[i]/Float64(di) : 0.0
            end
        end
    elseif init === :zeros
        # already zeros
        nothing
    else
        error("init must be :lu, :diag, or :zeros")
    end

    # GMRES work (reused)
    m   = restart
    V   = Array{inner}(undef, n, m+1)
    H   = Array{Float64}(undef, m+1, m)
    cs  = Array{Float64}(undef, m)
    sn  = Array{Float64}(undef, m)
    g   = Array{Float64}(undef, m+1)
    wlp = Array{inner}(undef, n)
    y   = Array{Float64}(undef, m)
    dlp = Array{inner}(undef, n)
    rlp = Array{inner}(undef, n)

    # Stopping scale
    bnorm2 = norm(b64)
    Anorm1 = (scale === :opnorm) ? (opnorm(A,1) |> Float64) : 0.0

    history_relres = Float64[]
    matvecs = 0
    total_inner_iters = 0

    for k in 0:maxiters
        # r64 = b - A*x (Float64, no allocs)
        mul!(Ax64, A, x64)
        @inbounds for i in 1:n
            r64[i] = b64[i] - Ax64[i]
        end

        # relative measure
        relres = if scale === :opnorm
            denom = Anorm1*maximum(abs, x64) + bnorm2
            (denom == 0.0) ? 0.0 : (norm(r64)/denom)
        else
            nb = (bnorm2 == 0.0) ? 1.0 : bnorm2
            norm(r64) / nb
        end
        push!(history_relres, relres)
        verbose && @info "[IR] iter=$(k)  rel=$(relres)"

        if relres ≤ tol || k == maxiters
            break
        end

        # Inner GMRES target: reduce residual aggressively so IR finishes in ≤ ~3 steps
        r2 = norm(r64)
        atol_inner = max(1e-4*r2, tol*bnorm2)          # stronger than before

        # cast r to low-precision
        @inbounds for i in 1:n
            rlp[i] = inner(r64[i])
        end

        # zero outputs
        fill!(dlp, zero(inner)); fill!(y, 0.0); fill!(g, 0.0)

        iters, _ = _gmres_lowprec!(A_lp, rlp, V, H, cs, sn, g, wlp, y, dlp, m, atol_inner)
        total_inner_iters += iters
        matvecs += iters

        # x64 += dlp (convert elementwise, no temp)
        @inbounds for i in 1:n
            x64[i] += Float64(dlp[i])
        end
    end

    stats = (relres = last(history_relres),
             history = history_relres,
             inner_iters = total_inner_iters,
             matvecs = matvecs,
             restart = restart,
             inner = inner,
             init = init,
             scale = scale)

    return x64, stats
end


gmres_ir

In [35]:
b = randn(size(A, 1))
x2, stats = gmres_ir(A, b,inner=Float32, tol=1e-12, maxiters=6, restart=30, verbose=true, init = :diag)
x1 = A \ b
println(norm(x1 .- x2)/norm(x1))

[ Info: [IR] iter=0  rel=1.3411396848963766
[ Info: [IR] iter=1  rel=0.2838316538740406
[ Info: [IR] iter=2  rel=0.193865393702251
[ Info: [IR] iter=3  rel=0.15933193091848843
[ Info: [IR] iter=4  rel=0.13999032775566045
[ Info: [IR] iter=5  rel=0.1279313023480881
[ Info: [IR] iter=6  rel=0.12107268218761655


0.935963360204064


In [36]:
b = rand(size(A, 1))
x2, stats = gmres_ir(A, b,inner=Float32, tol=1e-12, maxiters=6, restart=30, verbose=true, init =:lu)
x1 = A \ b
println(norm(x1 .- x2)/norm(x1))

[ Info: [IR] iter=0  rel=1.058414157814595e-5
[ Info: [IR] iter=1  rel=2.22716496541338e-6
[ Info: [IR] iter=2  rel=1.540505237951062e-6
[ Info: [IR] iter=3  rel=1.1852760667107529e-6
[ Info: [IR] iter=4  rel=1.0464333388504995e-6
[ Info: [IR] iter=5  rel=9.082681475081033e-7
[ Info: [IR] iter=6  rel=8.089590407903177e-7


1.5764018769754682e-7


In [37]:
@btime gmres_ir($A, $b,inner=Float32, tol=1e-12, maxiters=6, restart=30, verbose=false);
@btime gmres_ir($A, $b,inner=Float32, tol=1e-12, maxiters=6, restart=30, verbose=false, init = :lu);
@btime $A \ $b;
@btime lu($A) \ $b;

  318.780 ms (309 allocations: 168.93 MiB)
  308.737 ms (309 allocations: 168.93 MiB)
  181.325 ms (87 allocations: 151.94 MiB)
  180.043 ms (85 allocations: 151.76 MiB)


## Iteration 4

In [39]:
using LinearAlgebra
using SparseArrays

# ---------------- utilities: Float64 dot/norm (no @simd) ----------------------

@inline function dot64(x::AbstractVector{<:AbstractFloat}, y::AbstractVector{<:AbstractFloat})
    s = 0.0
    @inbounds for i in eachindex(x, y)
        s += Float64(x[i]) * Float64(y[i])
    end
    return s
end

@inline nrm2_64(x::AbstractVector{<:AbstractFloat}) = sqrt(dot64(x, x))

# ---------------- low-precision GMRES (restart) with MGS-64 -------------------

function _gmres_lowprec!(A_lp::AbstractMatrix{Ti},
                         r_lp::AbstractVector{Ti},
                         V::AbstractMatrix{Ti},         # n×(m+1)
                         H::AbstractMatrix{Float64},     # (m+1)×m
                         cs::AbstractVector{Float64},
                         sn::AbstractVector{Float64},
                         g::AbstractVector{Float64},     # length m+1
                         w_lp::AbstractVector{Ti},       # work (n)
                         y::AbstractVector{Float64},     # length m
                         d_lp::AbstractVector{Ti},       # output (n)
                         restart::Int,
                         atol::Float64) where {Ti<:AbstractFloat}

    m = restart
    fill!(H, 0.0); fill!(cs, 0.0); fill!(sn, 0.0); fill!(g, 0.0)

    β = nrm2_64(r_lp)
    if β == 0.0
        fill!(d_lp, zero(Ti))
        return 0, 0.0
    end
    @inbounds @views V[:,1] .= r_lp ./ Ti(β)
    g[1] = β

    k_done = 0
    resn = β

    for k in 1:m
        mul!(w_lp, A_lp, @view V[:,k])                 # w = A*v_k  (low-prec)

        # MGS with 64-bit dots
        @inbounds for j in 1:k
            hj = dot64(@view(V[:,j]), w_lp)
            H[j,k] = hj
            @views w_lp .-= Ti(hj) .* V[:,j]
        end
        hk1 = nrm2_64(w_lp)
        H[k+1,k] = hk1

        if hk1 != 0.0
            @views V[:,k+1] .= w_lp ./ Ti(hk1)
        else
            @views V[:,k+1] .= zero(Ti)
        end

        # apply old Givens
        @inbounds for j in 1:k-1
            c = cs[j]; s = sn[j]
            h1 = H[j,k]; h2 = H[j+1,k]
            H[j,k]   =  c*h1 + s*h2
            H[j+1,k] = -s*h1 + c*h2
        end
        # new Givens
        hkk  = H[k,k]; hk1k = H[k+1,k]
        den = hypot(hkk, hk1k) + eps(Float64)
        c = hkk/den; s = hk1k/den
        cs[k] = c; sn[k] = s
        H[k,k]   =  c*hkk + s*hk1k
        H[k+1,k] =  0.0

        # update g
        gk  = g[k]; gk1 = g[k+1]
        g[k]   =  c*gk + s*gk1
        g[k+1] = -s*gk + c*gk1

        resn = abs(g[k+1])
        k_done = k
        if resn ≤ atol
            break
        end
    end

    # backsolve R*y = g(1:k_done)
    @inbounds for i in k_done:-1:1
        ssum = 0.0
        @inbounds for j in i+1:k_done
            ssum += H[i,j] * y[j]
        end
        y[i] = (g[i] - ssum) / H[i,i]
    end

    # d = V(:,1:k_done) * y  (accumulate in low precision)
    fill!(d_lp, zero(Ti))
    @inbounds for j in 1:k_done
        α = Ti(y[j])
        @views d_lp .+= α .* V[:,j]
    end

    return k_done, resn
end

# ----------- x0 builder: a few Float64 Jacobi–Richardson steps ----------------

"""
Perform s steps of x ← x + ω * D^{-1} (b - A*x) in Float64 to build a robust x0.
This is *only* for initialization (not a preconditioner for GMRES).
- D is the diagonal of A (pre-extracted once).
"""
function _x0_richardson!(x64::Vector{Float64}, A::AbstractMatrix, b64::Vector{Float64},
                         D::Vector{Float64}, s::Int, ω::Float64,
                         Ax64::Vector{Float64}, r64::Vector{Float64})
    n = length(x64)
    for _ in 1:s
        mul!(Ax64, A, x64)                       # Ax
        @inbounds for i in 1:n
            r64[i] = b64[i] - Ax64[i]            # r = b - Ax
        end
        @inbounds for i in 1:n
            di = D[i]
            x64[i] += (di != 0.0) ? ω * (r64[i] / di) : 0.0
        end
    end
    return nothing
end

# ---------------- mixed-precision IR (sparse-friendly) ------------------------

"""
    x, stats = gmres_ir(A, b; inner=Float32, tol=1e-12,
                        maxiters=10, restart=50,
                        init=:richardson, x0_steps=4, x0_omega=0.9,
                        verbose=false)

Mixed-precision IR with low-precision GMRES and a robust Float64 x₀ builder.

Key choices:
- Outer arithmetic & residuals in Float64 (`mul!`, no allocs).
- Inner GMRES in `inner` (Float32/Float16) with **Float64 Arnoldi dots**.
- x₀ builder (no preconditioning):
    * `:richardson` (default): s steps of Jacobi–Richardson in Float64.
    * `:lu`        : one low-precision sparse LU solve to form x₀.
    * `:diag`/`:zeros` also available.

Returns `(x::Vector{Float64}, stats::NamedTuple)`.
"""
function gmres_ir(A, b; inner::Type{<:AbstractFloat}=Float32,
                  tol::Float64=1e-12, maxiters::Int=10, restart::Int=50,
                  init::Symbol=:richardson, x0_steps::Int=4, x0_omega::Float64=0.9,
                  verbose::Bool=false)

    n = size(A,1)
    @assert size(A,2) == n  "A must be square"
    @assert length(b) == n  "b has incompatible length"
    @assert restart ≥ 1

    # Float64 work
    b64 = Array{Float64}(undef, n); copyto!(b64, b)
    x64 = zeros(Float64, n)
    r64 = similar(b64)
    Ax64 = similar(b64)

    # Low-precision operator (preserve sparsity)
    A_lp = issparse(A) ? convert(SparseMatrixCSC{inner,Int}, A) : Array{inner}(A)

    # Diagonal (Float64) for x0 builder; extract once
    D = Vector{Float64}(undef, n)
    if issparse(A)
        Di = diag(A)
        @inbounds for i in 1:n
            D[i] = Float64(Di[i])
        end
    else
        @inbounds for i in 1:n
            D[i] = Float64(A[i,i])
        end
    end

    # Build x0
    if init === :lu
        F = lu(A_lp)                                # factorize ONCE to seed x0
        b_lp = Vector{inner}(undef, n); @inbounds b_lp .= inner.(b64)
        x0_lp = F \ b_lp
        @inbounds for i in 1:n
            x64[i] = Float64(x0_lp[i])
        end
    elseif init === :richardson
        _x0_richardson!(x64, A, b64, D, x0_steps, x0_omega, Ax64, r64)
    elseif init === :diag
        @inbounds for i in 1:n
            di = D[i]
            x64[i] = (di != 0.0) ? b64[i]/di : 0.0
        end
    elseif init === :zeros
        # already zeros
        nothing
    else
        error("init must be :richardson, :lu, :diag, or :zeros")
    end

    # GMRES work (reused)
    m   = restart
    V   = Array{inner}(undef, n, m+1)
    H   = Array{Float64}(undef, m+1, m)
    cs  = Array{Float64}(undef, m)
    sn  = Array{Float64}(undef, m)
    g   = Array{Float64}(undef, m+1)
    wlp = Array{inner}(undef, n)
    y   = Array{Float64}(undef, m)
    dlp = Array{inner}(undef, n)
    rlp = Array{inner}(undef, n)

    bnorm2 = norm(b64)
    history_relres = Float64[]
    matvecs = 0
    total_inner_iters = 0
    eps_inner = eps(float(inner))

    for k in 0:maxiters
        # r64 = b - A*x (Float64, no allocs)
        mul!(Ax64, A, x64)
        @inbounds for i in 1:n
            r64[i] = b64[i] - Ax64[i]
        end

        relres = (bnorm2 == 0.0) ? norm(r64) : norm(r64) / bnorm2
        push!(history_relres, relres)
        verbose && @info "[IR] iter=$(k)  rel=$(relres)"

        if relres ≤ tol || k == maxiters
            break
        end

        # Inner GMRES target tied to Float32/Float16 limits
        r2 = norm(r64)
        atol_inner = max(1e-2*r2, 100*eps_inner*r2)

        # cast r to low-precision
        @inbounds for i in 1:n
            rlp[i] = inner(r64[i])
        end

        # zero outputs
        fill!(dlp, zero(inner)); fill!(y, 0.0); fill!(g, 0.0); fill!(H, 0.0)

        iters, _ = _gmres_lowprec!(A_lp, rlp, V, H, cs, sn, g, wlp, y, dlp, m, atol_inner)
        total_inner_iters += iters
        matvecs += iters

        # x64 += dlp (convert elementwise, no temp)
        @inbounds for i in 1:n
            x64[i] += Float64(dlp[i])
        end
    end

    stats = (relres = last(history_relres),
             history = history_relres,
             inner_iters = total_inner_iters,
             matvecs = matvecs,
             restart = restart,
             inner = inner,
             init = init,
             x0_steps = x0_steps,
             x0_omega = x0_omega)

    return x64, stats
end

gmres_ir

In [40]:
# robust, no-LU path
@btime gmres_ir($A, $b, inner=Float32, tol=1e-12,
                maxiters=10, restart=50, init=:richardson, x0_steps=4, x0_omega=0.9, verbose=false);

# accuracy & convergence detail
x_ir, st = gmres_ir(A, b, inner=Float32, tol=1e-12,
                    maxiters=10, restart=50, init=:richardson, x0_steps=4, x0_omega=0.9, verbose=true)
x_ref = A \ b
println("rel error = ", norm(x_ref .- x_ir)/norm(x_ref))
println("final relres(IR) = ", st.relres, "  inner iters = ", st.inner_iters)

  458.504 ms (559 allocations: 12.16 MiB)


[ Info: [IR] iter=0  rel=16.770041401139522
[ Info: [IR] iter=1  rel=0.9443243518107844
[ Info: [IR] iter=2  rel=0.9441275771550612
[ Info: [IR] iter=3  rel=0.9440836171480682
[ Info: [IR] iter=4  rel=0.9440608431299026
[ Info: [IR] iter=5  rel=0.9440558686562239
[ Info: [IR] iter=6  rel=0.9440544268238439
[ Info: [IR] iter=7  rel=0.9440525476726951
[ Info: [IR] iter=8  rel=0.9440509655677307
[ Info: [IR] iter=9  rel=0.9440501453815676
[ Info: [IR] iter=10  rel=0.9440494373521084


rel error = 0.993366857193621
final relres(IR) = 0.9440494373521084  inner iters = 500


In [44]:
b = rand(size(A, 1))
x2, stats = gmres_ir(A, b,inner=Float64, tol=√eps(Float64), maxiters=6, restart=30, verbose=true, init =:lu)
x1 = A \ b
println(norm(x1 .- x2)/norm(x1))

[ Info: [IR] iter=0  rel=7.325605780206208e-13


0.0


In [45]:
b = rand(size(A, 1))
x2, stats = gmres_ir(A, b,inner=Float32, tol=√eps(Float64), maxiters=100, restart=30, verbose=true, init =:lu)
x1 = A \ b
println(norm(x1 .- x2)/norm(x1))

[ Info: [IR] iter=0  rel=1.0351441574178052e-5
[ Info: [IR] iter=1  rel=2.182514989193979e-6
[ Info: [IR] iter=2  rel=1.5079866499363195e-6
[ Info: [IR] iter=3  rel=1.163199475026662e-6
[ Info: [IR] iter=4  rel=1.024198794042988e-6
[ Info: [IR] iter=5  rel=8.881815711254513e-7
[ Info: [IR] iter=6  rel=7.926738282132527e-7
[ Info: [IR] iter=7  rel=7.234753787445355e-7
[ Info: [IR] iter=8  rel=6.599937767161819e-7
[ Info: [IR] iter=9  rel=6.217199567722353e-7
[ Info: [IR] iter=10  rel=5.955834127074243e-7
[ Info: [IR] iter=11  rel=5.687839687546161e-7
[ Info: [IR] iter=12  rel=5.361703211526774e-7
[ Info: [IR] iter=13  rel=5.06752037679742e-7
[ Info: [IR] iter=14  rel=4.837265068014272e-7
[ Info: [IR] iter=15  rel=4.5582174389570673e-7
[ Info: [IR] iter=16  rel=3.9955986757851287e-7
[ Info: [IR] iter=17  rel=3.068573609752965e-7
[ Info: [IR] iter=18  rel=2.4528163738141173e-7
[ Info: [IR] iter=19  rel=2.082527610271763e-7
[ Info: [IR] iter=20  rel=1.8279410335775495e-7
[ Info: [IR] iter=

5.228193395484042e-9


In [46]:
b = rand(size(A, 1))
x2, stats = gmres_ir(A, b,inner=Float16, tol=tol=√eps(Float64), maxiters=100, restart=30, verbose=true, init =:lu)
x1 = A \ b
println(norm(x1 .- x2)/norm(x1))

[ Info: [IR] iter=0  rel=0.08708234970962125
[ Info: [IR] iter=1  rel=0.03466976410470817
[ Info: [IR] iter=2  rel=0.02278635236593329
[ Info: [IR] iter=3  rel=0.01647175969676335
[ Info: [IR] iter=4  rel=0.013235168572149094
[ Info: [IR] iter=5  rel=0.010146401697070824
[ Info: [IR] iter=6  rel=0.008305503083698262
[ Info: [IR] iter=7  rel=0.007117754388302331
[ Info: [IR] iter=8  rel=0.006183801586517588
[ Info: [IR] iter=9  rel=0.005684485024823252
[ Info: [IR] iter=10  rel=0.005224080258645982
[ Info: [IR] iter=11  rel=0.0045582886067010026
[ Info: [IR] iter=12  rel=0.0038647642191373763
[ Info: [IR] iter=13  rel=0.0034977934451391407
[ Info: [IR] iter=14  rel=0.0032495513882540993
[ Info: [IR] iter=15  rel=0.0030398612123100927
[ Info: [IR] iter=16  rel=0.0028189347205915894
[ Info: [IR] iter=17  rel=0.0025996757009751894
[ Info: [IR] iter=18  rel=0.0023941126731482874
[ Info: [IR] iter=19  rel=0.002169900754432784
[ Info: [IR] iter=20  rel=0.0019954515418943784
[ Info: [IR] iter=

5.31425305530682e-6
